# Task 4: Genetic Algorithm v2.0 - Proper Evolutionary Experiment

## 🎯 Objective

This is a **CORRECTED** implementation that:
1. Starts with **NAIVE AI text** (no Victorian hints)
2. Uses GA to **DISCOVER** what fools detector through evolution
3. Tests detector robustness against evolutionary attacks
4. Provides meaningful findings regardless of success/failure

## 🔑 Key Differences from Failed v1.0

| Aspect | v1.0 (Failed) | v2.0 (Proper) |
|--------|---------------|---------------|
| **Initial Prompts** | Explicit Victorian instructions | Generic prompts (no hints) |
| **Expected Initial Fitness** | High (96.99% immediately) | Low (10-25% - detector catches) |
| **Mutation Strategy** | N/A (no evolution occurred) | 8 guided strategies (direction, not answers) |
| **Learning** | Trivial bypass (instruction following) | Genuine discovery through evolution |
| **Scientific Value** | Unexpected finding (more valuable) | Tests detector robustness |

## 📊 Expected Outcomes (All Valuable!)

- **Success (85%+ Human)**: Detector vulnerable → needs adversarial training
- **Partial Success (60-84%)**: Detector moderately robust → some weaknesses identified
- **Failure (<60%)**: Detector highly robust → production-ready (pending red-team testing)

**Author**: Research Team  
**Date**: February 2026

# 🔬 ABLATION STUDY OVERVIEW

## 🎯 Research Question
**Can blind mutations (no style guidance) discover Victorian patterns through pure evolution, or do we need domain expertise?**

## 📊 Three Conditions

### CONDITION A: BLIND MUTATIONS (True Evolution)
- **Word2Vec synonym replacement** - semantic similarity only, no style preference
- **Random sentence shuffling** - no logic about flow/chronology
- **Back-translation** (EN→FR→EN) - natural noise, not style-directed
- **Random word deletion** (10%) - no awareness of modern tells
- **Character typos** (1-2 per paragraph) - pure noise

✅ **Truly blind:** No LLM, no style instructions, no Victorian knowledge

### CONDITION B: GUIDED MUTATIONS (Current Approach)
- **Your existing 8 strategies** with Gemini
- Temporal shift, vocabulary elevation, structural complexity, etc.
- Mutations have direction ("make formal", "use past tense")

✅ **Domain-guided:** Uses Victorian knowledge implicitly

### CONDITION C: HYBRID (50/50 Mix)
- **50% blind, 50% guided** (random choice per mutation)
- Tests if combining approaches exploits detector better

## 🔍 What We'll Learn

| Outcome | Interpretation | Implication |
|---------|---------------|-------------|
| **Blind succeeds** | Detector has true vulnerabilities | Any random perturbations fool it → needs adversarial training |
| **Only Guided succeeds** | Detector robust to blind search | Only vulnerable to domain experts → decent robustness |
| **Hybrid is best** | Multiple weak points | Detector vulnerable to combined approach |
| **All fail** | Detector highly robust | Production-ready (pending red-team) |

## 📦 Experimental Design

**Controlled Variables (Same Across All):**
- Same initial population (10 generic AI paragraphs)
- Same fitness function (DistilBERT-LoRA detector)
- Same selection (top-3 elitism)
- Same hyperparameters (10 generations, 10 individuals)

**Independent Variable (What Changes):**
- **ONLY** mutation strategy (blind vs guided vs hybrid)

**Statistical Validity:**
- 3 random seeds per condition (42, 123, 456)
- Total: 9 experiments
- ANOVA + post-hoc tests for comparison

## 📈 Deliverables

**Plots:**
1. Fitness evolution (line plot with mean ± std)
2. Final distribution (boxplot, 3 conditions)
3. Victorian marker heatmaps (3 separate, one per condition)

**Statistical Analysis:**
- Mean ± std final fitness per condition
- ANOVA (do conditions differ?)
- Tukey's HSD post-hoc (which pairs differ?)
- Cohen's d effect sizes

**Data Files:**
- `ablation_raw_data.csv` - All generation data
- `convergence_analysis.csv` - Generations to 70% Human
- `best_individuals_all_conditions.txt` - Final evolved texts

---

---

# 🚀 Setup: Mount Drive & Install Dependencies

---

# ⚙️ Configuration

---

# 🔑 Configure Gemini API

In [1]:
# ==============================================================================
# 1. MASTER SETUP & ROBUST CONFIGURATION
# ==============================================================================
import os
import sys
import json
import torch
import random
import numpy as np
import pandas as pd
from datetime import datetime
from google.colab import drive, userdata

# A. Mount Drive with Forced Check
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive Mounted.")
else:
    print("✅ Drive already mounted.")

# B. Define Critical Paths (Centralized Source of Truth)
BASE_PATH = "/content/drive/MyDrive/precog"
PATHS = {
    "model_dir": f"{BASE_PATH}/lora_distilbert/lora_adapter",  # Your trained model
    "cache_dir": f"{BASE_PATH}/ablation_cache",                # Storage for Word2Vec/MarianMT
    "output_dir": f"{BASE_PATH}/task4_outputs",                # Results storage
    "checkpoint_file": f"{BASE_PATH}/task4_outputs/ga_checkpoint.json" # STATE SAVER
}

# C. Recursive Directory Check & Creation
for key, path in PATHS.items():
    if key == "checkpoint_file": continue # Don't create the file itself, just dir
    if not os.path.exists(path):
        print(f"📁 Creating missing directory: {path}")
        os.makedirs(path, exist_ok=True)
    else:
        print(f"✅ Found directory: {path}")

# D. Global Config
CONFIG = {
    "pop_size": 10,
    "generations": 10,
    "top_k": 3,
    "target_fitness": 0.85,
    "seed": 42
}

# F. Model Configuration
BASE_MODEL = "distilbert-base-uncased"  # Base model for LoRA
MAX_LENGTH = 512  # Tokenization max length

# G. GA-Specific Config
NUM_GENERATIONS = CONFIG["generations"]
POPULATION_SIZE = CONFIG["pop_size"]
TOP_K_SELECTION = CONFIG["top_k"]
TARGET_FITNESS = CONFIG["target_fitness"]

# H. Output Directory (from PATHS)
OUTPUT_DIR = PATHS["output_dir"]

print(f"⚙️ Model: {BASE_MODEL}")
print(f"⚙️ GA Config: {NUM_GENERATIONS} gens, {POPULATION_SIZE} pop, target={TARGET_FITNESS*100:.0f}%")

# E. Device Config
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️ Running on: {DEVICE}")

Mounted at /content/drive
✅ Drive Mounted.
✅ Found directory: /content/drive/MyDrive/precog/lora_distilbert/lora_adapter
✅ Found directory: /content/drive/MyDrive/precog/ablation_cache
✅ Found directory: /content/drive/MyDrive/precog/task4_outputs
⚙️ Model: distilbert-base-uncased
⚙️ GA Config: 10 gens, 10 pop, target=85%
⚙️ Running on: cpu


In [2]:
# 1. INSTALL LIBRARY (Required for fresh Colab sessions)
# ======================================================
print("📦 Installing/Updating Google Generative AI library...")
!pip install -q -U google-generativeai

# 2. IMPORTS
# ======================================================
import google.generativeai as genai
import socket
import sys
from google.colab import userdata

# 3. CONFIGURATION & EXECUTION
# ======================================================
print("\n🔐 Attempting to load Gemini API key from Colab Secrets...")

try:
    # Get API key from Colab Secrets
    try:
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
        if GEMINI_API_KEY is None:
            raise ValueError("Key is None")
        GEMINI_API_KEY = GEMINI_API_KEY.strip()
        print("✅ Successfully loaded API key from Colab Secrets!")

    except Exception as e:
        print("\n" + "❌" * 40)
        print("\n🔑 ERROR: Gemini API key not found in Colab Secrets!")
        print("\n👉 TO FIX THIS:")
        print("   1. Click the 🔑 key icon in the LEFT SIDEBAR")
        print("   2. Click '+ Add new secret'")
        print("   3. Name: GEMINI_API_KEY")
        print("   4. Value: Paste your API key from aistudio.google.com")
        print("   5. Toggle 'Notebook access' to ON")
        print("   6. Re-run this cell")
        print("\n❌" * 40)
        raise ValueError("Gemini API key not found. Please follow instructions above.")

except ImportError:
    print("\n⚠️  WARNING: Not running in Google Colab")
    raise RuntimeError("This notebook must be run in Google Colab")

# Configure Gemini
try:
    print("\n🧪 Configuring Gemini API...")
    genai.configure(api_key=GEMINI_API_KEY)

    # Updated to gemini-pro-latest (more stable than 1.5-flash)
    MODEL_NAME = 'gemini-pro-latest'

    generation_config = genai.GenerationConfig(
        temperature=0.4,
        top_p=0.8,
        top_k=40,
        max_output_tokens=200,
    )

    gemini_model = genai.GenerativeModel(
        MODEL_NAME,
        generation_config=generation_config
    )

    print(f"🧪 Testing connection to {MODEL_NAME} (15s timeout)...")

    # Set global socket timeout
    socket.setdefaulttimeout(15)

    try:
        # Generate content
        test_response = gemini_model.generate_content("Say 'API working!'")

        print("\n" + "="*80)
        print("✅ GEMINI API CONFIGURED SUCCESSFULLY!")
        print("="*80)
        print(f"🤖 Model: {MODEL_NAME}")
        print(f"🧪 Test response: {test_response.text.strip()}")
        print(f"🔒 API key loaded securely")
        print("="*80)

    except socket.timeout:
        print("\n" + "⏱️" * 40)
        print("\n⏱️  TIMEOUT ERROR: API request took >15 seconds")
        print("   - The free tier might be momentarily overloaded.")
        print("   - Try running the cell again in 1 minute.")
        print("\n⏱️" * 40)
        raise

except Exception as e:
    print("\n" + "❌" * 40)
    print(f"\n❌ Error configuring/testing Gemini API: {e}")
    print("\n💡 SOLUTION:")
    print("   1. Check your API key at aistudio.google.com")
    print("   2. Ensure 'Notebook access' is enabled in the Secrets tab")
    print("   3. Restart Runtime (Runtime > Restart Session)")
    print("\n❌" * 40)
    raise

📦 Installing/Updating Google Generative AI library...


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



🔐 Attempting to load Gemini API key from Colab Secrets...
✅ Successfully loaded API key from Colab Secrets!

🧪 Configuring Gemini API...
🧪 Testing connection to gemini-pro-latest (15s timeout)...

✅ GEMINI API CONFIGURED SUCCESSFULLY!
🤖 Model: gemini-pro-latest
🧪 Test response: API working
🔒 API key loaded securely


---

# 🔍 Search for Trained Model

In [3]:
# Search for your trained model
import os
import glob

print("🔍 Searching for trained LoRA model...")
print("=" * 80)

# Common locations where the model might be
search_paths = [
    "/content/lora_distilbert",  # Default local location
    "/content/reports/lora_distilbert",
    "/content/lora_adapter",
    "/content/drive/MyDrive/precog/lora_distilbert",
    "/home/avani/precog/reports/lora_distilbert",  # Local machine path
]

found_models = []

for path in search_paths:
    if os.path.exists(path):
        # Check if it has adapter files
        adapter_config = os.path.join(path, "lora_adapter", "adapter_config.json")
        adapter_model = os.path.join(path, "lora_adapter", "adapter_model.safetensors")

        if os.path.exists(adapter_config) or os.path.exists(adapter_model):
            found_models.append(os.path.join(path, "lora_adapter"))
            print(f"✅ FOUND: {os.path.join(path, 'lora_adapter')}")
        elif os.path.exists(os.path.join(path, "adapter_config.json")):
            found_models.append(path)
            print(f"✅ FOUND: {path}")

# Also search recursively
print(f"\n🔍 Searching recursively in /content...")
recursive_search = glob.glob("/content/**/adapter_config.json", recursive=True)
for config_path in recursive_search:
    model_dir = os.path.dirname(config_path)
    if model_dir not in found_models:
        found_models.append(model_dir)
        print(f"✅ FOUND: {model_dir}")

print("\n" + "=" * 80)
if found_models:
    print(f"\n🎉 Found {len(found_models)} trained model(s)!")
    print(f"\n💡 Update MODEL_DIR in the configuration cell to:")
    for model_path in found_models:
        print(f'   MODEL_DIR = "{model_path}"')

    # Auto-set to first found model
    # Set MODEL_DIR globally for later cells
    global MODEL_DIR
    MODEL_DIR = found_models[0]
    print(f"\n✅ Automatically set MODEL_DIR to: {MODEL_DIR}")
else:
    print("\n❌ No trained model found!")
    print("\n⚠️  This means:")
    print("   1. The model was trained in a previous Colab session (storage deleted)")
    print("   2. You need to retrain the model")
    print("   3. OR download from GitHub/Drive if you saved it externally")
    print("\n💡 To avoid losing models in future:")
    print("   - Mount Google Drive BEFORE training")
    print("   - Save model to Drive: /content/drive/MyDrive/precog/")
    MODEL_DIR = None

🔍 Searching for trained LoRA model...
✅ FOUND: /content/drive/MyDrive/precog/lora_distilbert/lora_adapter

🔍 Searching recursively in /content...
✅ FOUND: /content/drive/MyDrive/precog/lora_distilbert/checkpoint-175
✅ FOUND: /content/drive/MyDrive/precog/lora_distilbert/checkpoint-350


🎉 Found 3 trained model(s)!

💡 Update MODEL_DIR in the configuration cell to:
   MODEL_DIR = "/content/drive/MyDrive/precog/lora_distilbert/lora_adapter"
   MODEL_DIR = "/content/drive/MyDrive/precog/lora_distilbert/checkpoint-175"
   MODEL_DIR = "/content/drive/MyDrive/precog/lora_distilbert/checkpoint-350"

✅ Automatically set MODEL_DIR to: /content/drive/MyDrive/precog/lora_distilbert/lora_adapter


---

# 📚 Import Libraries

In [4]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import re
from datetime import datetime
from typing import List, Dict, Tuple
from collections import defaultdict

# ML imports
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# Gemini
import google.generativeai as genai

# Readability
try:
    import textstat
    TEXTSTAT_AVAILABLE = True
except ImportError:
    print("⚠️  textstat not installed. Readability scoring will be disabled.")
    TEXTSTAT_AVAILABLE = False

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

⚠️  textstat not installed. Readability scoring will be disabled.
✅ All libraries imported successfully!


---

# 🧬 Phase 2: Smart Mutation Strategies

These mutations provide **DIRECTION** but don't give away the answer.

In [5]:
MUTATION_STRATEGIES = [
    # Strategy 1: Temporal shift (nudges toward past tense)
    {
        'name': 'temporal_shift',
        'prompt': """Rewrite this paragraph changing the primary time frame.
If mostly present tense, shift some verbs to past. If mostly future,
shift to past or present. Vary the temporal perspective.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Shifts temporal perspective (nudges toward past tense)'
    },

    # Strategy 2: Vocabulary elevation (nudges toward richer words)
    {
        'name': 'vocabulary_elevation',
        'prompt': """Rewrite this paragraph using more sophisticated, literary
vocabulary. Replace common words with rarer synonyms. Make it sound
more literary and less conversational, like older literature.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Elevates vocabulary (nudges toward Victorian diction)'
    },

    # Strategy 3: Structural complexity (nudges toward complex syntax)
    {
        'name': 'structural_complexity',
        'prompt': """Rewrite this paragraph with more complex sentence structures.
Combine short sentences, use subordinate clauses, add semicolons or
em-dashes for sophisticated punctuation. Make sentences flow together.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Increases structural complexity (Victorian syntax)'
    },

    # Strategy 4: Narrative voice shift (nudges toward first-person)
    {
        'name': 'narrative_voice',
        'prompt': """Rewrite this paragraph experimenting with narrative perspective.
If third-person, try first-person observer. If impersonal, add a narrator's
voice. If already first-person, strengthen the personal perspective.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Changes narrative voice (nudges toward first-person)'
    },

    # Strategy 5: Formality adjustment (nudges away from modern casual)
    {
        'name': 'formality_increase',
        'prompt': """Rewrite this paragraph in a more formal, old-fashioned style.
Remove contractions, avoid modern colloquialisms, use more formal
conjunctions and transitions. Make it sound like classic literature.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Increases formality (nudges toward Victorian register)'
    },

    # Strategy 6: Descriptive density (nudges toward Victorian detail)
    {
        'name': 'descriptive_enhancement',
        'prompt': """Rewrite this paragraph adding more sensory details and
atmospheric description. Focus on visual imagery, sounds, smells,
textures. Create vivid, immersive scene-setting.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Adds sensory details (Victorian atmospheric style)'
    },

    # Strategy 7: Rhythm variation (nudges away from AI uniformity)
    {
        'name': 'rhythm_variation',
        'prompt': """Rewrite this paragraph varying sentence length and rhythm.
Mix short, punchy sentences with longer, flowing ones. Create natural
variation in pacing. Avoid uniformity in structure.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Varies rhythm (breaks AI uniformity patterns)'
    },

    # Strategy 8: Connective reworking (nudges away from "however")
    {
        'name': 'transition_rework',
        'prompt': """Rewrite this paragraph changing how ideas connect. Replace
common transitions (however, therefore, additionally, furthermore) with
different ways to link thoughts. Use varied conjunctions.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Reworks transitions (avoids modern AI tells)'
    }
]

print("🧬 MUTATION STRATEGIES:")
print("=" * 80)
print(f"Total strategies: {len(MUTATION_STRATEGIES)}\n")
for i, strategy in enumerate(MUTATION_STRATEGIES, 1):
    print(f"{i}. {strategy['name']:<25s}: {strategy['description']}")
print("\n🎯 Key Design Principle:")
print("   Mutations provide DIRECTION (more formal, past tense, complex)")
print("   But DON'T specify exact markers (ere, lest, etc.)")
print("   GA must DISCOVER which directions improve fitness.")
print("=" * 80)

🧬 MUTATION STRATEGIES:
Total strategies: 8

1. temporal_shift           : Shifts temporal perspective (nudges toward past tense)
2. vocabulary_elevation     : Elevates vocabulary (nudges toward Victorian diction)
3. structural_complexity    : Increases structural complexity (Victorian syntax)
4. narrative_voice          : Changes narrative voice (nudges toward first-person)
5. formality_increase       : Increases formality (nudges toward Victorian register)
6. descriptive_enhancement  : Adds sensory details (Victorian atmospheric style)
7. rhythm_variation         : Varies rhythm (breaks AI uniformity patterns)
8. transition_rework        : Reworks transitions (avoids modern AI tells)

🎯 Key Design Principle:
   Mutations provide DIRECTION (more formal, past tense, complex)
   But DON'T specify exact markers (ere, lest, etc.)
   GA must DISCOVER which directions improve fitness.


---

# 🔄 Phase 4: Adaptive Mutation Selector

In [6]:
class AdaptiveMutationSelector:
    """
    Tracks which mutation strategies are most successful and adapts selection.

    Early generations: Random exploration
    Later generations: Favor successful strategies
    """

    def __init__(self, strategies: List[Dict]):
        self.strategies = strategies
        self.success_counts = {s['name']: 0 for s in strategies}
        self.attempt_counts = {s['name']: 0 for s in strategies}
        self.success_rates = {s['name']: 0.0 for s in strategies}
        self.fitness_improvements = {s['name']: [] for s in strategies}

    def select_strategy(self, generation: int) -> Dict:
        """Select mutation strategy based on generation and success history."""
        if generation <= 3:
            # Exploration phase: Random selection
            return random.choice(self.strategies)
        else:
            # Exploitation phase: Weighted by success rate
            weights = [self.success_rates.get(s['name'], 0.0) + 0.1
                      for s in self.strategies]
            return random.choices(self.strategies, weights=weights)[0]

    def record_mutation(self, strategy_name: str, parent_fitness: float,
                       child_fitness: float):
        """Track whether mutation improved fitness."""
        self.attempt_counts[strategy_name] += 1

        improvement = child_fitness - parent_fitness
        self.fitness_improvements[strategy_name].append(improvement)

        if child_fitness > parent_fitness:
            self.success_counts[strategy_name] += 1

        # Update success rate
        if self.attempt_counts[strategy_name] > 0:
            self.success_rates[strategy_name] = (
                self.success_counts[strategy_name] /
                self.attempt_counts[strategy_name]
            )

    def get_statistics(self) -> pd.DataFrame:
        """Get mutation strategy effectiveness statistics."""
        stats = []
        for strategy in self.strategies:
            name = strategy['name']
            stats.append({
                'strategy': name,
                'attempts': self.attempt_counts[name],
                'successes': self.success_counts[name],
                'success_rate': self.success_rates[name],
                'avg_improvement': np.mean(self.fitness_improvements[name])
                                  if self.fitness_improvements[name] else 0.0,
                'total_improvement': sum(self.fitness_improvements[name])
            })

        df = pd.DataFrame(stats)
        df = df.sort_values('success_rate', ascending=False)
        return df

    def print_statistics(self):
        """Print formatted mutation strategy effectiveness report."""
        print("\n" + "="*80)
        print("MUTATION STRATEGY EFFECTIVENESS:")
        print("="*80)

        df = self.get_statistics()

        print(f"\n{'Strategy':<25s} {'Attempts':>8s} {'Successes':>10s} "
              f"{'Success Rate':>13s} {'Avg Δ Fitness':>15s}")
        print("-"*80)

        for _, row in df.iterrows():
            print(f"{row['strategy']:<25s} {row['attempts']:>8.0f} "
                  f"{row['successes']:>10.0f} {row['success_rate']:>12.1%} "
                  f"{row['avg_improvement']:>+14.4f}")

        print("\n📈 TOP 3 MOST EFFECTIVE:")
        for i, (_, row) in enumerate(df.head(3).iterrows(), 1):
            print(f"   #{i}: {row['strategy']} "
                  f"({row['success_rate']:.1%} success rate, "
                  f"{row['attempts']:.0f} attempts)")

print("✅ Adaptive mutation selector class defined!")

✅ Adaptive mutation selector class defined!


---

# 🤖 Load DistilBERT-LoRA Model

---

# 🔬 ABLATION STUDY: Install Additional Dependencies

For Condition A (Blind Mutations), we need Word2Vec, MarianMT, and spaCy.

In [11]:
# ==============================================================================
# 2. SMART CACHED LOADER (Fixes Re-downloading)
# ==============================================================================
# Install lightweight libs every time (fast)
!pip install -q transformers[torch] peft accelerate datasets google-generativeai textstat gensim spacy sentencepiece

import spacy
import gensim.downloader as api
from gensim.models import KeyedVectors
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def load_heavy_dependencies(cache_path):
    print("\n🔄 Starting Smart Dependency Load...")

    # 1. SpaCy (Lightweight check)
    if not spacy.util.is_package("en_core_web_sm"):
        print("📥 Downloading spaCy model...")
        spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy loaded.")

    # 2. Word2Vec (Heavy - 1.6GB Recursive Check)
    w2v_path = os.path.join(cache_path, "word2vec-google-news-300")

    if os.path.exists(w2v_path):
        print(f"💾 Found cached Word2Vec in Drive. Loading from disk (Fast)...")
        # mmap='r' keeps RAM usage low by reading directly from Drive
        word2vec = KeyedVectors.load(w2v_path, mmap='r')
    else:
        print(f"⚠️ Cache miss. Downloading Word2Vec (This happens ONCE)...")
        word2vec = api.load('word2vec-google-news-300')
        print(f"💾 Saving Word2Vec to Drive for future runs...")
        word2vec.save(w2v_path)

    print("✅ Word2Vec loaded.")
    return nlp, word2vec

# Execute Smart Load
nlp_model, word2vec_model = load_heavy_dependencies(PATHS["cache_dir"])
# Initialize BlindMutationEngine now that dependencies are loaded
print("\n🤖 Initializing Blind Mutation Engine...")
blind_engine = BlindMutationEngine(word2vec_model=word2vec_model, device='cpu')
print("✅ Blind Mutation Engine initialized!")



🔄 Starting Smart Dependency Load...
✅ spaCy loaded.
💾 Found cached Word2Vec in Drive. Loading from disk (Fast)...
✅ Word2Vec loaded.

🤖 Initializing Blind Mutation Engine...
📚 Loading spaCy model...
🔄 Loading MarianMT models...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

✅ BlindMutationEngine initialized!
✅ Blind Mutation Engine initialized!


Run above cell once, then define Blind engine using below cell and re run above cell to initialize

---

# 🧬 CONDITION A: Blind Mutation Engine

**TRUE EVOLUTION** - No style awareness, no LLM guidance.

In [12]:
import spacy
from transformers import MarianMTModel, MarianTokenizer

class BlindMutationEngine:
    """
    Condition A: Blind mutations with NO style awareness.

    Uses:
    - Word2Vec synonym replacement (semantic similarity only)
    - Random sentence shuffling
    - Back-translation (English → French → English)
    - Random word deletion (10% of words)
    - Character-level typos (1-2 per paragraph)

    NO EXPLICIT STYLE INSTRUCTIONS - mutations don't "know" about Victorian patterns.
    """

    def __init__(self, word2vec_model, device='cpu'):
        self.word2vec = word2vec_model
        self.device = device

        # Load spaCy for NER (to avoid deleting named entities)
        print("📚 Loading spaCy model...")
        self.nlp = spacy.load('en_core_web_sm')

        # Load MarianMT for back-translation
        print("🔄 Loading MarianMT models...")
        self.tokenizer_en_fr = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-fr')
        self.model_en_fr = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-fr').to(device)

        self.tokenizer_fr_en = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-fr-en')
        self.model_fr_en = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-fr-en').to(device)

        print("✅ BlindMutationEngine initialized!")

    def mutate(self, text: str) -> Tuple[str, str]:
        """
        Apply ONE random blind mutation.

        Returns:
            (mutated_text, mutation_type)
        """
        mutation_type = random.choice([
            'synonym_replacement',
            'sentence_shuffle',
            'back_translate',
            'word_deletion',
            'character_noise'
        ])

        mutated_text = self._apply_mutation(text, mutation_type)
        return mutated_text, mutation_type

    def _apply_mutation(self, text: str, mutation_type: str) -> str:
        """Apply specific mutation type."""

        if mutation_type == 'synonym_replacement':
            return self._synonym_replacement(text)
        elif mutation_type == 'sentence_shuffle':
            return self._sentence_shuffle(text)
        elif mutation_type == 'back_translate':
            return self._back_translate(text)
        elif mutation_type == 'word_deletion':
            return self._word_deletion(text)
        elif mutation_type == 'character_noise':
            return self._character_noise(text)
        else:
            return text

    def _synonym_replacement(self, text: str, num_replacements: int = 3) -> str:
        """Replace 3-5 random words with Word2Vec most_similar synonyms."""
        words = text.split()
        if len(words) < 5:
            return text

        # Get named entities to avoid replacing them
        doc = self.nlp(text)
        entity_words = set([token.text.lower() for ent in doc.ents for token in ent])

        replaceable_indices = [
            i for i, word in enumerate(words)
            if word.lower() not in entity_words and word.isalpha() and len(word) > 3
        ]

        if not replaceable_indices:
            return text

        num_to_replace = min(num_replacements, len(replaceable_indices))
        indices_to_replace = random.sample(replaceable_indices, num_to_replace)

        for idx in indices_to_replace:
            word = words[idx]
            try:
                # Get most similar words from Word2Vec (semantic similarity only)
                similar_words = self.word2vec.most_similar(word, topn=5)
                # Pick random synonym (not necessarily more formal/archaic)
                synonym = random.choice([w for w, _ in similar_words])

                # Preserve capitalization
                if word[0].isupper():
                    synonym = synonym.capitalize()

                words[idx] = synonym
            except KeyError:
                # Word not in vocabulary - skip
                pass

        return ' '.join(words)

    def _sentence_shuffle(self, text: str) -> str:
        """Randomly shuffle sentences."""
        # Split into sentences (simple split on .!?)
        sentences = re.split(r'([.!?]+)', text)

        # Combine sentences with their punctuation
        sentence_pairs = []
        for i in range(0, len(sentences) - 1, 2):
            if i + 1 < len(sentences):
                sentence_pairs.append(sentences[i] + sentences[i + 1])
            else:
                sentence_pairs.append(sentences[i])

        if len(sentence_pairs) <= 1:
            return text

        # Shuffle randomly (NO logic about chronology/flow)
        random.shuffle(sentence_pairs)

        return ' '.join(sentence_pairs).strip()

    def _back_translate(self, text: str) -> str:
        """
        English → French → English back-translation.
        Translation artifacts emerge naturally (not directed toward any style).
        """
        try:
            # English → French
            inputs_en = self.tokenizer_en_fr(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs_en = {k: v.to(self.device) for k, v in inputs_en.items()}

            translated_fr = self.model_en_fr.generate(**inputs_en, max_length=512)
            french_text = self.tokenizer_en_fr.batch_decode(translated_fr, skip_special_tokens=True)[0]

            # French → English
            inputs_fr = self.tokenizer_fr_en(french_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs_fr = {k: v.to(self.device) for k, v in inputs_fr.items()}

            translated_en = self.model_fr_en.generate(**inputs_fr, max_length=512)
            back_translated = self.tokenizer_fr_en.batch_decode(translated_en, skip_special_tokens=True)[0]

            return back_translated
        except Exception as e:
            print(f"   ⚠️ Back-translation failed: {e}")
            return text

    def _word_deletion(self, text: str, deletion_rate: float = 0.10) -> str:
        """
        Delete 10% of words randomly.
        Exclude named entities (spaCy NER).
        NO awareness of removing modern tells - just noise injection.
        """
        doc = self.nlp(text)
        entity_indices = set()

        for ent in doc.ents:
            for token in ent:
                entity_indices.add(token.i)

        words = []
        for i, token in enumerate(doc):
            if i not in entity_indices and random.random() > deletion_rate:
                words.append(token.text_with_ws)
            elif i in entity_indices:
                words.append(token.text_with_ws)

        return ''.join(words).strip()

    def _character_noise(self, text: str, num_typos: int = 2) -> str:
        """
        Introduce 1-2 character swaps/deletions per paragraph.
        Random noise, not strategic misspellings.
        """
        text_list = list(text)
        if len(text_list) < 20:
            return text

        # Get indices of alphabetic characters (avoid punctuation)
        alpha_indices = [i for i, c in enumerate(text_list) if c.isalpha()]

        if not alpha_indices:
            return text

        num_to_modify = min(num_typos, len(alpha_indices))
        indices_to_modify = random.sample(alpha_indices, num_to_modify)

        for idx in indices_to_modify:
            mutation_choice = random.choice(['swap', 'delete'])

            if mutation_choice == 'swap' and idx + 1 < len(text_list):
                # Swap adjacent characters
                text_list[idx], text_list[idx + 1] = text_list[idx + 1], text_list[idx]
            elif mutation_choice == 'delete':
                # Delete character
                text_list[idx] = ''

        return ''.join(text_list)

# Initialize blind mutation engine (will be used if running Condition A)
print("\n🔬 Initializing Blind Mutation Engine...")
try:
    blind_engine = BlindMutationEngine(word2vec_model, device=device)
    print("✅ Blind Mutation Engine ready!")
except Exception as e:
    print(f"⚠️ Could not initialize Blind Engine: {e}")
    print("   Run dependency installation cell first.")
    blind_engine = None


🔬 Initializing Blind Mutation Engine...
📚 Loading spaCy model...
🔄 Loading MarianMT models...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

✅ BlindMutationEngine initialized!
✅ Blind Mutation Engine ready!


---

# 🎯 CONDITION B: Guided Mutation Engine

**CURRENT APPROACH** - Uses your existing 8 strategies with Gemini.

In [14]:
class GuidedMutationEngine:
    """
    Condition B: Guided mutations using Gemini with style instructions.

    This is your CURRENT approach - wrapping existing mutation strategies.
    """

    def __init__(self, gemini_model, strategies, api_key, url, headers):
        self.gemini_model = gemini_model
        self.strategies = strategies
        self.selector = AdaptiveMutationSelector(strategies)
        self.api_key = api_key
        self.url = url
        self.headers = headers

    def mutate(self, text: str, generation: int, parent_fitness: float = None) -> Tuple[str, str]:
        """
        Apply guided mutation using Gemini.

        Returns:
            (mutated_text, strategy_name)
        """
        # Select strategy based on adaptive selection
        strategy = self.selector.select_strategy(generation)

        try:
            # Format prompt
            prompt = strategy['prompt'].format(text=text)

            # Call Gemini API
            data = {
                "contents": [{"parts": [{"text": prompt}]}],
                "safetySettings": [
                    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
                ],
                "generationConfig": {"temperature": 0.7, "maxOutputTokens": 8192}
            }

            response = requests.post(self.url, headers=self.headers,
                                   data=json.dumps(data), timeout=30)

            if response.status_code == 200:
                result = response.json()
                mutated_text = result['candidates'][0]['content']['parts'][0]['text']
            else:
                mutated_text = text  # Fallback

            return mutated_text, strategy['name']

        except Exception as e:
            print(f"   ⚠️ Guided mutation error: {e}")
            return text, strategy['name']

    def record_mutation(self, strategy_name: str, parent_fitness: float, child_fitness: float):
        """Record mutation effectiveness."""
        self.selector.record_mutation(strategy_name, parent_fitness, child_fitness)

    def get_statistics(self):
        """Get mutation statistics."""
        return self.selector.get_statistics()

    def print_statistics(self):
        """Print mutation statistics."""
        self.selector.print_statistics()

# Initialize guided engine with proper API configuration
print("🎯 Initializing Guided Mutation Engine...")
API_KEY = GEMINI_API_KEY
URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?key={API_KEY}"
HEADERS = {'Content-Type': 'application/json'}

guided_engine = GuidedMutationEngine(
    gemini_model=None,  # Not used with REST API
    strategies=MUTATION_STRATEGIES,
    api_key=API_KEY,
    url=URL,
    headers=HEADERS
)
print("✅ Guided Mutation Engine initialized!")


🎯 Initializing Guided Mutation Engine...
✅ Guided Mutation Engine initialized!


---

# ⚡ CONDITION C: Hybrid Mutation Engine

**50/50 MIX** - Randomly chooses blind or guided.

In [15]:
class HybridMutationEngine:
    """
    Condition C: 50/50 mix of blind and guided mutations.

    Randomly chooses which engine to use on each mutation.
    Tests if combining both approaches exploits detector better.
    """

    def __init__(self, blind_engine, guided_engine):
        self.blind = blind_engine
        self.guided = guided_engine

        # Track which type was used
        self.blind_count = 0
        self.guided_count = 0

    def mutate(self, text: str, generation: int, parent_fitness: float = None) -> Tuple[str, str]:
        """
        Randomly choose blind or guided mutation (50/50).

        Returns:
            (mutated_text, mutation_type)
        """
        if random.random() < 0.5:
            # Blind mutation
            mutated, mutation_type = self.blind.mutate(text)
            self.blind_count += 1
            return mutated, f'blind_{mutation_type}'
        else:
            # Guided mutation
            mutated, strategy_name = self.guided.mutate(text, generation, parent_fitness)
            self.guided_count += 1
            return mutated, f'guided_{strategy_name}'

    def record_mutation(self, mutation_name: str, parent_fitness: float, child_fitness: float):
        """Record mutation effectiveness."""
        # Route to appropriate engine
        if mutation_name.startswith('guided_'):
            strategy_name = mutation_name.replace('guided_', '')
            self.guided.record_mutation(strategy_name, parent_fitness, child_fitness)

    def get_statistics(self):
        """Get combined statistics."""
        return self.guided.get_statistics()  # For now, track guided stats

    def print_statistics(self):
        """Print statistics."""
        print(f"\n🔀 Hybrid Statistics:")
        print(f"   Blind mutations used:  {self.blind_count}")
        print(f"   Guided mutations used: {self.guided_count}")
        self.guided.print_statistics()

# Initialize hybrid engine
print("⚡ Initializing Hybrid Mutation Engine...")
if blind_engine is not None and guided_engine is not None:
    hybrid_engine = HybridMutationEngine(blind_engine, guided_engine)
    print("✅ Hybrid Mutation Engine ready!")
else:
    if blind_engine is None:
        print("⚠️ Blind engine not initialized - run dependency installation cell")
    if guided_engine is None:
        print("⚠️ Guided engine not initialized - run Phase 2 cells first")
    print("⚠️ Cannot initialize Hybrid Engine - need both Blind and Guided engines")
    hybrid_engine = None


⚡ Initializing Hybrid Mutation Engine...
✅ Hybrid Mutation Engine ready!


---

# 🚀 ABLATION STUDY: Experiment Harness

Run all 3 conditions × 3 seeds = 9 experiments.

In [16]:
from dataclasses import dataclass
from typing import List, Dict, Tuple
from datetime import datetime, timedelta

@dataclass
class ExperimentResult:
    """Results from one GA experiment."""
    condition: str
    seed: int
    history: List[Dict]
    final_population: List[Dict]
    best_individual: Dict
    runtime: timedelta
    mutation_stats: pd.DataFrame

def run_ablation_experiment(
    condition: str,
    seed: int,
    initial_population: List[str],
    num_generations: int = NUM_GENERATIONS,
    population_size: int = POPULATION_SIZE,
    top_k: int = TOP_K_SELECTION,
    target_fitness: float = TARGET_FITNESS
) -> ExperimentResult:
    """
    Run ONE complete GA experiment with specified condition.

    Args:
        condition: 'blind', 'guided', or 'hybrid'
        seed: Random seed for reproducibility
        initial_population: Starting population (SAME for all conditions)
        num_generations: Number of generations to run
        population_size: Population size
        top_k: Number of survivors per generation
        target_fitness: Target Human probability

    Returns:
        ExperimentResult with full experimental data
    """

    print("\n" + "="*80)
    print(f"🔬 RUNNING ABLATION EXPERIMENT")
    print("="*80)
    print(f"Condition: {condition.upper()}")
    print(f"Seed: {seed}")
    print(f"Initial population: {len(initial_population)} individuals")
    print(f"Generations: {num_generations}")
    print("="*80)

    # Set seed for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Select mutation engine
    if condition == 'blind':
        if blind_engine is None:
            raise ValueError("Blind engine not initialized! Run dependency installation cell.")
        engine = blind_engine
    elif condition == 'guided':
        if guided_engine is None:
            raise ValueError("Guided engine not initialized!")
        engine = guided_engine
    elif condition == 'hybrid':
        if hybrid_engine is None:
            raise ValueError("Hybrid engine not initialized!")
        engine = hybrid_engine
    else:
        raise ValueError(f"Unknown condition: {condition}")

    # Initialize
    population = initial_population.copy()
    history = []
    start_time = datetime.now()

    # GA Loop
    for generation in range(1, num_generations + 1):
        print(f"\n{'─'*80}")
        print(f"Generation {generation}/{num_generations} ({condition.upper()})")
        print(f"{'─'*80}")

        # Evaluate population
        evaluated = []
        for text in population:
            result = calculate_comprehensive_fitness(text, distilbert_predictor)
            result['text'] = text
            evaluated.append(result)

        evaluated = sorted(evaluated, key=lambda x: x['fitness'], reverse=True)

        # Track history
        best = evaluated[0]
        avg_fitness = np.mean([e['fitness'] for e in evaluated])

        history.append({
            'generation': generation,
            'best_fitness': best['fitness'],
            'best_human_prob': best['human_prob'],
            'avg_fitness': avg_fitness,
            'best_text': best['text'],
            'best_predicted_class': best['predicted_class']
        })

        print(f"   Best: {best['fitness']:.4f} ({best['human_prob']*100:.2f}% Human, {best['predicted_class']})")
        print(f"   Avg:  {avg_fitness:.4f}")

        # Check target
        if best['human_prob'] >= target_fitness:
            print(f"\n   🎉 Target achieved ({target_fitness*100:.0f}% Human)!")
            break

        # Selection
        survivors = [e['text'] for e in evaluated[:top_k]]
        survivor_fitness = [e['fitness'] for e in evaluated[:top_k]]

        if generation == num_generations:
            break

        # Mutation
        print(f"   🧬 Mutating...")
        next_population = survivors.copy()  # Elitism

        mutations_needed = population_size - top_k
        mutations_per_parent = max(1, mutations_needed // top_k)

        mutation_count = 0
        for parent_idx, (parent_text, parent_fitness) in enumerate(zip(survivors, survivor_fitness), 1):
            for mutation_idx in range(mutations_per_parent):
                if len(next_population) >= population_size:
                    break

                try:
                    # Apply mutation (blind, guided, or hybrid)
                    if condition == 'blind':
                        mutated_text, mutation_type = engine.mutate(parent_text)
                    else:
                        mutated_text, mutation_type = engine.mutate(parent_text, generation, parent_fitness)

                    # Evaluate child
                    child_result = calculate_comprehensive_fitness(mutated_text, distilbert_predictor)

                    # Record (for guided/hybrid)
                    if condition != 'blind':
                        engine.record_mutation(mutation_type, parent_fitness, child_result['fitness'])

                    next_population.append(mutated_text)
                    mutation_count += 1

                    # For blind mutations, add delay only for back-translation
                    if condition == 'guided' or (condition == 'hybrid' and 'guided' in mutation_type):
                        time.sleep(2.0)  # Rate limiting for Gemini
                    elif condition == 'blind' and mutation_type == 'back_translate':
                        time.sleep(0.5)  # Brief delay for MarianMT

                except Exception as e:
                    print(f"      ⚠️ Mutation error: {e}")
                    next_population.append(parent_text)

        # Fill remaining
        while len(next_population) < population_size:
            next_population.append(random.choice(survivors))

        population = next_population[:population_size]
        print(f"   ✓ Generation {generation + 1} ready")

    end_time = datetime.now()
    runtime = end_time - start_time

    # Final evaluation
    final_evaluated = []
    for text in population:
        result = calculate_comprehensive_fitness(text, distilbert_predictor)
        result['text'] = text
        final_evaluated.append(result)

    final_evaluated = sorted(final_evaluated, key=lambda x: x['fitness'], reverse=True)
    best_individual = final_evaluated[0]

    # Get mutation stats
    if condition != 'blind':
        mutation_stats = engine.get_statistics()
    else:
        mutation_stats = pd.DataFrame()  # Blind doesn't track strategy effectiveness

    print(f"\n{'='*80}")
    print(f"✅ EXPERIMENT COMPLETE ({condition.upper()})")
    print(f"   Final best: {best_individual['human_prob']*100:.2f}% Human")
    print(f"   Runtime: {runtime}")
    print(f"{'='*80}")

    return ExperimentResult(
        condition=condition,
        seed=seed,
        history=history,
        final_population=final_evaluated,
        best_individual=best_individual,
        runtime=runtime,
        mutation_stats=mutation_stats
    )

print("✅ Ablation experiment harness defined!")

✅ Ablation experiment harness defined!


---

# 🎬 RUN FULL ABLATION STUDY

**⚠️ WARNING:** This will take ~3.5 hours!
- Condition A (Blind): ~5 min × 3 seeds = ~15 min
- Condition B (Guided): ~45 min × 3 seeds = ~2.25 hours
- Condition C (Hybrid): ~25 min × 3 seeds = ~1.25 hours

Run this cell to execute all 9 experiments.

In [26]:
def run_full_ablation_study(
    initial_population: List[str],
    seeds: List[int] = [42, 123, 456],
    conditions: List[str] = ['blind', 'guided', 'hybrid']
) -> Dict[str, List[ExperimentResult]]:
    """
    Run complete ablation study: 3 conditions × 3 seeds = 9 experiments.

    Args:
        initial_population: Starting population (SAME for all)
        seeds: Random seeds for statistical validity
        conditions: Which conditions to run

    Returns:
        Dict mapping condition → list of results (one per seed)
    """

    print("\n" + "🔬"*40)
    print("🔬 STARTING FULL ABLATION STUDY")
    print("🔬"*40)
    print(f"\nConditions: {', '.join(c.upper() for c in conditions)}")
    print(f"Seeds: {seeds}")
    print(f"Total experiments: {len(conditions) * len(seeds)}")
    print(f"Initial population: {len(initial_population)} individuals")
    print("\n" + "🔬"*40)

    ablation_start = datetime.now()
    results = {condition: [] for condition in conditions}

    experiment_num = 0
    total_experiments = len(conditions) * len(seeds)

    for condition in conditions:
        print(f"\n\n{'='*80}")
        print(f"🧪 CONDITION: {condition.upper()}")
        print(f"{'='*80}")

        for seed in seeds:
            experiment_num += 1
            print(f"\n[Experiment {experiment_num}/{total_experiments}] " +
                  f"{condition.upper()} (seed={seed})")

            try:
                result = run_ablation_experiment(
                    condition=condition,
                    seed=seed,
                    initial_population=initial_population
                )
                results[condition].append(result)

                print(f"\n   ✅ Completed: {result.best_individual['human_prob']*100:.2f}% Human")

            except Exception as e:
                print(f"\n   ❌ ERROR: {e}")
                import traceback
                traceback.print_exc()
                continue

    ablation_end = datetime.now()
    ablation_runtime = ablation_end - ablation_start

    print("\n\n" + "🎉"*40)
    print("🎉 ABLATION STUDY COMPLETE!")
    print("🎉"*40)
    print(f"\nTotal runtime: {ablation_runtime}")
    print(f"Experiments completed: {sum(len(v) for v in results.values())}/{total_experiments}")
    print("\n" + "🎉"*40)

    return results

# Note: Don't run automatically - user can run when ready
print("✅ Full ablation study function defined!")
print("\n💡 To run ablation study:")
print("   ablation_results = run_full_ablation_study(")
print("       initial_population=[r['text'] for r in initial_results]")
print("   )")
print("\n⚠️  This will take ~3.5 hours!")

✅ Full ablation study function defined!

💡 To run ablation study:
   ablation_results = run_full_ablation_study(
       initial_population=[r['text'] for r in initial_results]
   )

⚠️  This will take ~3.5 hours!


In [18]:
# Define device first
device = device

print("🤖 Loading DistilBERT-LoRA model...")
print("=" * 80)

if MODEL_DIR is None or not os.path.exists(MODEL_DIR):
    print("❌ ERROR: Model directory not found!")
    print(f"   MODEL_DIR = {MODEL_DIR}")
    print("\n💡 Please run the 'Search for Trained Model' cell first.")
    raise FileNotFoundError(f"Model not found at {MODEL_DIR}")

try:
    # Load tokenizer
    print(f"Loading tokenizer from {MODEL_DIR}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    print("✅ Tokenizer loaded")

    # Load base model
    print(f"Loading base model: {BASE_MODEL}...")
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,
        id2label={0: "Human", 1: "AI"},
        label2id={"Human": 0, "AI": 1}
    )
    print("✅ Base model loaded")

    # Load LoRA adapter
    print(f"Loading LoRA adapter from {MODEL_DIR}...")
    model = PeftModel.from_pretrained(base_model, MODEL_DIR)
    print("✅ LoRA adapter loaded")

    # Move to device
    model = model.to(device)
    model.eval()
    print(f"✅ Model moved to {device}")

    print("\n" + "=" * 80)
    print("✅ DistilBERT-LoRA MODEL LOADED SUCCESSFULLY!")
    print("=" * 80)
    print(f"Base model: {BASE_MODEL}")
    print(f"LoRA adapter: {MODEL_DIR}")
    print(f"Device: {device}")
    print(f"Max length: {MAX_LENGTH}")
    print("=" * 80)

except Exception as e:
    print(f"\n❌ Error loading model: {e}")
    print("\n💡 Make sure the model directory contains:")
    print("   - adapter_config.json")
    print("   - adapter_model.safetensors (or adapter_model.bin)")
    print("   - tokenizer files (tokenizer_config.json, vocab.txt, etc.)")
    raise

🤖 Loading DistilBERT-LoRA model...
Loading tokenizer from /content/drive/MyDrive/precog/lora_distilbert/lora_adapter...
✅ Tokenizer loaded
Loading base model: distilbert-base-uncased...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Base model loaded
Loading LoRA adapter from /content/drive/MyDrive/precog/lora_distilbert/lora_adapter...


✅ LoRA adapter loaded
✅ Model moved to cpu

✅ DistilBERT-LoRA MODEL LOADED SUCCESSFULLY!
Base model: distilbert-base-uncased
LoRA adapter: /content/drive/MyDrive/precog/lora_distilbert/lora_adapter
Device: cpu
Max length: 512


---

# 🔮 Create Predictor Class

In [19]:
# ==============================================================================
# 3. STATE MANAGER (Fixes Resume Capability)
# ==============================================================================
class StateManager:
    def __init__(self, filepath):
        self.filepath = filepath

    def save_state(self, generation, population, history):
        """Saves exact state to Drive immediately."""
        state = {
            "last_completed_generation": generation,
            "population": population, # Current text population
            "history": history,       # Stats history
            "timestamp": str(datetime.now())
        }
        with open(self.filepath, 'w') as f:
            json.dump(state, f, indent=2)
        print(f"   💾 Progress saved to Drive (Gen {generation})")

    def load_state(self):
        """Checks for previous run to resume."""
        if os.path.exists(self.filepath):
            print(f"🔄 Found checkpoint at {self.filepath}")
            try:
                with open(self.filepath, 'r') as f:
                    state = json.load(f)
                return state
            except json.JSONDecodeError:
                print("⚠️ Checkpoint corrupted. Starting fresh.")
                return None
        return None

state_manager = StateManager(PATHS["checkpoint_file"])

In [20]:
class DistilBERTPredictor:
    """
    Predictor wrapper for DistilBERT-LoRA model.
    Provides predict_proba() method compatible with GA fitness function.
    """

    def __init__(self, model, tokenizer, device, max_length=512):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.max_length = max_length

    def predict_proba(self, texts: List[str]) -> np.ndarray:
        """
        Predict probabilities for list of texts.

        Args:
            texts: List of text strings

        Returns:
            numpy array of shape (n_samples, 2) with [prob_human, prob_ai]
        """
        self.model.eval()

        # Tokenize
        encodings = self.tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Move to device
        input_ids = encodings['input_ids'].to(self.device)
        attention_mask = encodings['attention_mask'].to(self.device)

        # Predict
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)

        return probs.cpu().numpy()

# Create predictor instance
distilbert_predictor = DistilBERTPredictor(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=MAX_LENGTH
)

print("✅ Predictor created successfully!")

# Test predictor
print("\n🧪 Testing predictor...")
test_text = "This is a test paragraph to verify the predictor works correctly."
test_probs = distilbert_predictor.predict_proba([test_text])[0]
print(f"   Test text: {test_text[:50]}...")
print(f"   Probabilities: {test_probs[0]*100:.2f}% Human, {test_probs[1]*100:.2f}% AI")
print(f"   Prediction: {'Human' if test_probs[0] > test_probs[1] else 'AI'}")
print("\n✅ Predictor working correctly!")

✅ Predictor created successfully!

🧪 Testing predictor...
   Test text: This is a test paragraph to verify the predictor w...
   Probabilities: 20.04% Human, 79.96% AI
   Prediction: AI

✅ Predictor working correctly!


---

# 📝 Phase 1: Naive Initial Prompts

These prompts are **GENERIC** with **NO Victorian hints**.

They should produce text that the detector **catches** (10-25% Human confidence).

In [21]:
NAIVE_INITIAL_PROMPTS = [
    # Generic mystery/detective prompts WITHOUT Victorian markers
    "Write a 100-150 word paragraph about a mysterious crime scene in London.",

    "Write a 100-150 word paragraph about a detective examining evidence and clues.",

    "Write a 100-150 word paragraph about an adventure on a sailing ship.",

    "Write a 100-150 word paragraph about a dark Gothic mansion at midnight.",

    "Write a 100-150 word paragraph about searching for hidden treasure.",

    "Write a 100-150 word paragraph about a scientific investigation of strange phenomena.",

    "Write a 100-150 word paragraph about local folklore and supernatural tales.",

    "Write a 100-150 word paragraph about an ancient family curse.",

    "Write a 100-150 word paragraph about the dual nature of human morality.",

    "Write a 100-150 word paragraph about a disturbed grave in an old cemetery."
]

print("📝 NAIVE INITIAL PROMPTS:")
print("=" * 80)
print(f"Total prompts: {len(NAIVE_INITIAL_PROMPTS)}")
print("\nKey characteristics:")
print("  ✓ Generic topics (mystery, detective, Gothic, etc.)")
print("  ✓ NO Victorian style instructions")
print("  ✓ NO archaic conjunction hints (ere, lest, thence)")
print("  ✓ NO past tense requirements")
print("  ✓ NO first-person narrative guidance")
print("  ✓ NO 'avoid however' instructions")
print("\n🎯 Expected: Detector CATCHES these (10-25% Human confidence)")
print("   This forces GA to genuinely evolve to improve fitness.")
print("=" * 80)

📝 NAIVE INITIAL PROMPTS:
Total prompts: 10

Key characteristics:
  ✓ Generic topics (mystery, detective, Gothic, etc.)
  ✓ NO Victorian style instructions
  ✓ NO archaic conjunction hints (ere, lest, thence)
  ✓ NO past tense requirements
  ✓ NO first-person narrative guidance
  ✓ NO 'avoid however' instructions

🎯 Expected: Detector CATCHES these (10-25% Human confidence)
   This forces GA to genuinely evolve to improve fitness.


---

# 🎯 Phase 3: Multi-Objective Fitness Function

In [22]:
def calculate_comprehensive_fitness(text: str, distilbert_predictor) -> dict:
    """
    Multi-objective fitness that rewards:
    1. Fooling the detector (PRIMARY - 100% weight)
    2. Maintaining readability (SECONDARY - 10% bonus)
    3. Preserving length (TERTIARY - 5% bonus)

    Args:
        text: Text to evaluate
        distilbert_predictor: DistilBERT predictor with predict_proba method

    Returns:
        Dict with fitness score and component metrics
    """
    # Primary objective: Fool the detector
    probs = distilbert_predictor.predict_proba([text])[0]
    human_prob = probs[0]
    ai_prob = probs[1]

    # Secondary objective: Readability (don't become gibberish)
    readability_bonus = 0.0
    flesch_score = None

    if TEXTSTAT_AVAILABLE:
        try:
            flesch_score = textstat.flesch_reading_ease(text)
            # Victorian texts: Flesch ~40-60 (difficult but readable)
            if 30 <= flesch_score <= 70:
                readability_bonus = 0.10  # 10% bonus for good readability
            elif 20 <= flesch_score < 30 or 70 < flesch_score <= 80:
                readability_bonus = 0.05  # 5% bonus for acceptable
            else:
                readability_bonus = 0.0  # No bonus if too easy or too hard
        except:
            flesch_score = None
            readability_bonus = 0.0

    # Tertiary objective: Length consistency (100-150 words)
    word_count = len(text.split())
    if 100 <= word_count <= 150:
        length_bonus = 0.05  # 5% bonus for perfect length
    elif 80 <= word_count <= 170:
        length_bonus = 0.02  # 2% bonus for acceptable length
    else:
        length_bonus = 0.0  # No bonus if too short/long

    # Combined fitness (max possible: 1.15)
    total_fitness = human_prob + readability_bonus + length_bonus

    return {
        'fitness': total_fitness,
        'human_prob': human_prob,
        'ai_prob': ai_prob,
        'flesch_score': flesch_score,
        'word_count': word_count,
        'readability_bonus': readability_bonus,
        'length_bonus': length_bonus,
        'predicted_class': 'Human' if human_prob > ai_prob else 'AI'
    }

print("✅ Multi-objective fitness function defined!")
print("\n🎯 Fitness Components:")
print("   1. Primary (100%):   Fool detector (Human probability)")
print("   2. Secondary (10%):  Maintain readability (Flesch 30-70)")
print("   3. Tertiary (5%):    Preserve length (100-150 words)")
print("   Max fitness: 1.15")

✅ Multi-objective fitness function defined!

🎯 Fitness Components:
   1. Primary (100%):   Fool detector (Human probability)
   2. Secondary (10%):  Maintain readability (Flesch 30-70)
   3. Tertiary (5%):    Preserve length (100-150 words)
   Max fitness: 1.15


---

# 📊 Phase 5: Victorian Marker Analysis

In [23]:
def count_victorian_markers(text: str) -> dict:
    """
    Count Victorian authenticity markers in text.

    These markers were identified in XAI analysis but are NOT
    explicitly told to the initial prompts. GA must DISCOVER them.
    """
    text_lower = text.lower()

    # Archaic conjunctions (Victorian tells)
    archaic_conj = len(re.findall(
        r'\b(ere|lest|thence|whence|wherefore|whilst)\b',
        text_lower
    ))

    # Past tense markers
    past_tense = len(re.findall(
        r'\b(was|were|had)\b',
        text_lower
    ))

    # First-person pronouns
    first_person = len(re.findall(
        r'\b(i|he|she|we)\b',
        text_lower
    ))

    # Modern transitions (AI tells - should decrease)
    modern_trans = len(re.findall(
        r'\b(however|therefore|additionally|furthermore|moreover)\b',
        text_lower
    ))

    # Complex punctuation
    semicolons = text.count(';')
    em_dashes = text.count('—') + text.count(' - ')

    # Victorian vocabulary (sample indicators)
    victorian_vocab = len(re.findall(
        r'\b(sepulchral|miasma|ghastly|singular|devilry|aghast|'
        r'thence|ere|lest|wherefore|whence|whilst)\b',
        text_lower
    ))

    return {
        'archaic_conj': archaic_conj,
        'past_tense': past_tense,
        'first_person': first_person,
        'modern_trans': modern_trans,
        'semicolons': semicolons,
        'em_dashes': em_dashes,
        'victorian_vocab': victorian_vocab
    }

def analyze_victorian_markers_evolution(history: List[Dict]) -> pd.DataFrame:
    """Track how Victorian markers emerge over generations."""
    evolution_data = []

    for gen in history:
        markers = count_victorian_markers(gen['best_text'])
        evolution_data.append({
            'generation': gen['generation'],
            'fitness': gen['best_fitness'],
            'human_prob': gen['best_human_prob'],
            **markers
        })

    return pd.DataFrame(evolution_data)

def plot_victorian_markers_evolution(df: pd.DataFrame, output_path: str):
    """Create visualization of Victorian marker emergence."""
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    axes = axes.flatten()

    metrics = [
        ('human_prob', 'Human Probability', 'green'),
        ('archaic_conj', 'Archaic Conjunctions (ere, lest)', 'blue'),
        ('past_tense', 'Past Tense Markers (was, had)', 'orange'),
        ('first_person', 'First-Person Pronouns (I, he)', 'red'),
        ('modern_trans', 'Modern Transitions (however)', 'purple'),
        ('semicolons', 'Semicolons', 'brown'),
        ('em_dashes', 'Em-Dashes', 'pink'),
        ('victorian_vocab', 'Victorian Vocabulary', 'teal')
    ]

    for i, (metric, title, color) in enumerate(metrics):
        ax = axes[i]
        ax.plot(df['generation'], df[metric], marker='o', linewidth=2.5,
               markersize=8, color=color, label=title)
        ax.set_xlabel('Generation', fontsize=11, fontweight='bold')
        ax.set_ylabel('Count' if metric != 'human_prob' else 'Probability',
                     fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(df['generation'].min() - 0.5, df['generation'].max() + 0.5)

        # Add trend line
        z = np.polyfit(df['generation'], df[metric], 1)
        p = np.poly1d(z)
        ax.plot(df['generation'], p(df['generation']),
               linestyle='--', alpha=0.5, color='gray', linewidth=1.5)

    # Hide unused subplot
    axes[-1].axis('off')

    plt.suptitle('Victorian Marker Evolution Across Generations',
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.99])

    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {output_path}")
    plt.close()

print("✅ Victorian marker analysis functions defined!")

✅ Victorian marker analysis functions defined!


---

# 🚀 Phase 6: Generate Initial Population

This uses the **naive prompts** with Gemini API.

In [24]:
import requests
import json

# Store the prompts for later use
INITIAL_PROMPTS = NAIVE_INITIAL_PROMPTS

print("=" * 80)
print("🚀 GENERATING INITIAL POPULATION")
print("=" * 80)
print(f"Creating {len(INITIAL_PROMPTS)} paragraphs using GENERIC prompts...")
print("Expected: Low fitness (10-25% Human) - detector catches them")
print("=" * 80)

# Configuration
API_KEY = GEMINI_API_KEY
URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?key={API_KEY}"
HEADERS = {'Content-Type': 'application/json'}

initial_population = []

for i, prompt in enumerate(INITIAL_PROMPTS, 1):
    print(f"\n[{i}/{len(INITIAL_PROMPTS)}] Generating...", end=" ")

    # JSON Payload with MAXIMUM limits
    data = {
        "contents": [{
            "parts": [{"text": prompt}]
        }],
        "safetySettings": [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
        ],
        "generationConfig": {
            "temperature": 0.7,
            "maxOutputTokens": 8192
        }
    }

    try:
        response = requests.post(
            URL,
            headers=HEADERS,
            data=json.dumps(data),
            timeout=30
        )

        if response.status_code == 200:
            result = response.json()
            try:
                candidate = result['candidates'][0]
                text = candidate['content']['parts'][0]['text']

                word_count = len(text.split())
                print(f"✅ Success ({word_count} words)")
                initial_population.append(text)

            except (KeyError, IndexError):
                finish_reason = result.get('candidates', [{}])[0].get('finishReason', 'UNKNOWN')
                print(f"⚠️ Empty. Reason: {finish_reason}")
                fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
                initial_population.append(fallback)
        else:
            print(f"❌ Status {response.status_code}: {response.text}")
            fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
            initial_population.append(fallback)

    except requests.exceptions.Timeout:
        print("❌ TIMEOUT (Even after 30s!)")
        fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
        initial_population.append(fallback)
    except Exception as e:
        print(f"❌ Error: {e}")
        fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
        initial_population.append(fallback)

    # Rate limiting
    time.sleep(2.0)

print(f"\n{'='*80}")
print(f"📝 Initial Population: {len(initial_population)} paragraphs generated")
print(f"{'='*80}")

🚀 GENERATING INITIAL POPULATION
Creating 10 paragraphs using GENERIC prompts...
Expected: Low fitness (10-25% Human) - detector catches them

[1/10] Generating... ✅ Success (136 words)

[2/10] Generating... ✅ Success (133 words)

[3/10] Generating... ✅ Success (113 words)

[4/10] Generating... ✅ Success (121 words)

[5/10] Generating... ✅ Success (125 words)

[6/10] Generating... ✅ Success (123 words)

[7/10] Generating... ✅ Success (123 words)

[8/10] Generating... ✅ Success (125 words)

[9/10] Generating... ✅ Success (133 words)

[10/10] Generating... ✅ Success (134 words)

📝 Initial Population: 10 paragraphs generated


---

# 📊 Evaluate Initial Population

In [25]:
print("=" * 80)
print("EVALUATING INITIAL POPULATION")
print("=" * 80)

initial_results = []
for i, text in enumerate(initial_population, 1):
    result = calculate_comprehensive_fitness(text, distilbert_predictor)
    result['text'] = text
    initial_results.append(result)
    print(f"[{i}/{len(initial_population)}] Fitness: {result['fitness']:.4f} "
          f"({result['human_prob']*100:.2f}% Human, {result['predicted_class']})")

initial_results = sorted(initial_results, key=lambda x: x['fitness'], reverse=True)

# Analysis
print("\n" + "="*80)
print("INITIAL POPULATION ANALYSIS:")
print("="*80)
print(f"Best fitness:    {initial_results[0]['fitness']:.4f} "
      f"({initial_results[0]['human_prob']*100:.2f}% Human)")
print(f"Worst fitness:   {initial_results[-1]['fitness']:.4f} "
      f"({initial_results[-1]['human_prob']*100:.2f}% Human)")
print(f"Average fitness: {np.mean([r['fitness'] for r in initial_results]):.4f} "
      f"({np.mean([r['human_prob'] for r in initial_results])*100:.2f}% Human)")
print(f"Std dev:         {np.std([r['fitness'] for r in initial_results]):.4f}")

if initial_results[0]['human_prob'] > 0.35:
    print("\n⚠️  WARNING: Initial population already has >35% Human confidence!")
    print("   The prompts may contain hints. Evolution may be trivial.")
else:
    print("\n✅ GOOD: Initial population has low Human confidence")
    print("   Detector catches them. GA will need to genuinely evolve!")
    print("   This tests: Can evolution discover Victorian patterns from scratch?")

# Save initial population
initial_df = pd.DataFrame([{
    'text': r['text'],
    'fitness': r['fitness'],
    'human_prob': r['human_prob'],
    'ai_prob': r['ai_prob'],
    'predicted_class': r['predicted_class'],
    'word_count': r['word_count']
} for r in initial_results])
initial_df.to_csv(f"{OUTPUT_DIR}/initial_population.csv", index=False)
print(f"\n✅ Saved: {OUTPUT_DIR}/initial_population.csv")

EVALUATING INITIAL POPULATION
[1/10] Fitness: 0.0506 (0.06% Human, AI)
[2/10] Fitness: 0.0502 (0.02% Human, AI)
[3/10] Fitness: 0.0504 (0.04% Human, AI)
[4/10] Fitness: 0.0500 (0.00% Human, AI)
[5/10] Fitness: 0.0501 (0.01% Human, AI)
[6/10] Fitness: 0.0502 (0.02% Human, AI)
[7/10] Fitness: 0.0501 (0.01% Human, AI)
[8/10] Fitness: 0.0501 (0.01% Human, AI)
[9/10] Fitness: 0.0501 (0.01% Human, AI)
[10/10] Fitness: 0.0502 (0.02% Human, AI)

INITIAL POPULATION ANALYSIS:
Best fitness:    0.0506 (0.06% Human)
Worst fitness:   0.0500 (0.00% Human)
Average fitness: 0.0502 (0.02% Human)
Std dev:         0.0002

✅ GOOD: Initial population has low Human confidence
   Detector catches them. GA will need to genuinely evolve!
   This tests: Can evolution discover Victorian patterns from scratch?

✅ Saved: /content/drive/MyDrive/precog/task4_outputs/initial_population.csv


---

# 🧬 Main Genetic Algorithm Loop

This is where the **actual evolution** happens!

In [ ]:
# ==============================================================================
# 4. ROBUST EXPERIMENT EXECUTION LOOP
# ==============================================================================

def run_resumable_experiment():
    # 1. Check for Resume
    saved_state = state_manager.load_state()

    if saved_state:
        start_gen = saved_state["last_completed_generation"] + 1
        population = saved_state["population"]
        history = saved_state["history"]
        print(f"\n⏩ RESUMING experiment from Generation {start_gen}...")
    else:
        print("\n🚀 STARTING NEW experiment...")
        start_gen = 1
        # Use your existing logic to generate initial prompts
        population = generate_initial_population(NAIVE_INITIAL_PROMPTS)
        history = []

    # 2. The Main Loop
    for generation in range(start_gen, CONFIG["generations"] + 1):
        print(f"\n{'='*60}\n🧬 GENERATION {generation}/{CONFIG['generations']}\n{'='*60}")

        # --- A. Evaluation ---
        evaluated = []
        for text in population:
            # Using your existing fitness function
            metrics = calculate_comprehensive_fitness(text, distilbert_predictor)
            metrics['text'] = text
            evaluated.append(metrics)

        # Sort by fitness (descending)
        evaluated = sorted(evaluated, key=lambda x: x['fitness'], reverse=True)
        best = evaluated[0]

        # --- B. Logging & Checkpointing ---
        stats = {
            "generation": generation,
            "best_fitness": float(best['fitness']),
            "best_human_prob": float(best['human_prob']),
            "avg_fitness": float(sum(e['fitness'] for e in evaluated) / len(evaluated)),
            "best_text_snippet": best['text'][:100]
        }
        history.append(stats)

        print(f"   📊 Best Human Prob: {best['human_prob']*100:.2f}% | Avg Fitness: {stats['avg_fitness']:.4f}")

        # CRITICAL: SAVE STATE NOW
        state_manager.save_state(generation, [e['text'] for e in evaluated], history)

        # --- C. Target Check ---
        if best['human_prob'] >= CONFIG["target_fitness"]:
            print(f"\n🎉 Target Achieved! ({best['human_prob']:.2%})")
            break

        # --- D. Selection & Mutation (Skip if last gen) ---
        if generation < CONFIG["generations"]:
            survivors = [e['text'] for e in evaluated[:CONFIG["top_k"]]]
            next_pop = survivors.copy()

            print("   🧬 Mutating...")
            while len(next_pop) < CONFIG["pop_size"]:
                parent = random.choice(survivors)
                # Use your existing mutation selector
                strategy = mutation_selector.select_strategy(generation)

                try:
                    # Use your existing engine wrapper
                    # NOTE: Ensure 'guided_engine' is initialized before this loop!
                    mutated_text, strat_name = guided_engine.mutate(parent, generation, best['fitness'])
                    next_pop.append(mutated_text)
                except Exception as e:
                    print(f"      ⚠️ Mutation failed: {e}")
                    next_pop.append(parent) # Fallback to parent

            population = next_pop

    return history

# 3. RUN
history = run_resumable_experiment()


🚀 STARTING NEW experiment...


NameError: name 'generate_initial_population' is not defined

---

# 📊 Final Analysis & Results

In [ ]:
# Final evaluation
final_evaluated = []
for text in population:
    result = calculate_comprehensive_fitness(text, distilbert_predictor)
    result['text'] = text
    final_evaluated.append(result)

final_evaluated = sorted(final_evaluated, key=lambda x: x['fitness'], reverse=True)
final_best = final_evaluated[0]

print("=" * 80)
print("FINAL EXPERIMENTAL REPORT")
print("=" * 80)

print(f"\n⏱️  Runtime: {elapsed}")
print(f"\n📈 EVOLUTION SUMMARY:")
print(f"   Initial best:  {initial_results[0]['human_prob']*100:.2f}% Human")
print(f"   Final best:    {final_best['human_prob']*100:.2f}% Human")
print(f"   Improvement:   {(final_best['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%")
print(f"   Generations:   {len(history)}")
print(f"   Target ({TARGET_FITNESS*100:.0f}%): "
      f"{'✅ ACHIEVED' if final_best['human_prob'] >= TARGET_FITNESS else '❌ NOT ACHIEVED'}")

# Mutation effectiveness
mutation_selector.print_statistics()

# Victorian markers
print("\n" + "="*80)
print("VICTORIAN MARKER EMERGENCE:")
print("="*80)

markers_df = analyze_victorian_markers_evolution(history)

initial_markers = count_victorian_markers(initial_results[0]['text'])
final_markers = count_victorian_markers(final_best['text'])

print("\nMarker Evolution (Initial → Final):")
for marker in ['archaic_conj', 'past_tense', 'first_person', 'modern_trans', 'victorian_vocab']:
    initial_val = initial_markers[marker]
    final_val = final_markers[marker]
    change = final_val - initial_val
    symbol = '⬆️' if change > 0 else '⬇️' if change < 0 else '➡️'
    print(f"   {marker:20s}: {initial_val:3d} → {final_val:3d} ({change:+3d}) {symbol}")

# Plot
plot_victorian_markers_evolution(markers_df, f"{OUTPUT_DIR}/victorian_markers_evolution.png")

# Save results
print("\n💾 Saving results...")

history_df = pd.DataFrame(history)
history_df.to_csv(f"{OUTPUT_DIR}/evolution_history.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/evolution_history.csv")

markers_df.to_csv(f"{OUTPUT_DIR}/victorian_markers_evolution.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/victorian_markers_evolution.csv")

mutation_stats_df = mutation_selector.get_statistics()
mutation_stats_df.to_csv(f"{OUTPUT_DIR}/mutation_strategy_stats.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/mutation_strategy_stats.csv")

final_df = pd.DataFrame([{
    'text': e['text'],
    'fitness': e['fitness'],
    'human_prob': e['human_prob'],
    'predicted_class': e['predicted_class']
} for e in final_evaluated])
final_df.to_csv(f"{OUTPUT_DIR}/final_population.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/final_population.csv")

with open(f"{OUTPUT_DIR}/best_evolved_text.txt", 'w') as f:
    f.write("="*80 + "\n")
    f.write("BEST EVOLVED TEXT\n")
    f.write("="*80 + "\n\n")
    f.write(f"Fitness: {final_best['fitness']:.4f}\n")
    f.write(f"Human Probability: {final_best['human_prob']:.4f} ({final_best['human_prob']*100:.2f}%)\n")
    f.write(f"Predicted Class: {final_best['predicted_class']}\n")
    f.write(f"Generations: {len(history)}\n")
    f.write(f"Success: {'YES' if final_best['human_prob'] >= TARGET_FITNESS else 'NO'}\n")
    f.write("\n" + "="*80 + "\n\n")
    f.write(final_best['text'])
    f.write("\n\n" + "="*80 + "\n")
print(f"   ✓ {OUTPUT_DIR}/best_evolved_text.txt")

print("\n✅ EXPERIMENT COMPLETE!")
print(f"📁 All results saved to: {OUTPUT_DIR}/")

In [ ]:
import spacy
from transformers import MarianMTModel, MarianTokenizer

class BlindMutationEngine:
    """
    Condition A: Blind mutations with NO style awareness.

    Uses:
    - Word2Vec synonym replacement (semantic similarity only)
    - Random sentence shuffling
    - Back-translation (English → French → English)
    - Random word deletion (10% of words)
    - Character-level typos (1-2 per paragraph)

    NO EXPLICIT STYLE INSTRUCTIONS - mutations don't "know" about Victorian patterns.
    """

    def __init__(self, word2vec_model, device='cpu', cache_dir=None):
        self.word2vec = word2vec_model
        self.device = device

        # Load spaCy for NER (to avoid deleting named entities)
        print("📚 Loading spaCy model...")
        self.nlp = spacy.load('en_core_web_sm')

        # Load MarianMT for back-translation (with caching)
        print("🔄 Loading MarianMT models...")

        if cache_dir and os.path.exists(cache_dir):
            # Try to load from cache
            en_fr_cache = f"{cache_dir}/marian_en_fr"
            fr_en_cache = f"{cache_dir}/marian_fr_en"

            if os.path.exists(en_fr_cache) and os.path.exists(fr_en_cache):
                print("   ✅ Loading MarianMT from cache...")
                self.tokenizer_en_fr = MarianTokenizer.from_pretrained(en_fr_cache)
                self.model_en_fr = MarianMTModel.from_pretrained(en_fr_cache).to(device)

                self.tokenizer_fr_en = MarianTokenizer.from_pretrained(fr_en_cache)
                self.model_fr_en = MarianMTModel.from_pretrained(fr_en_cache).to(device)
                print("   ✅ Loaded from cache!")
            else:
                # Download and cache
                print("   📥 Downloading MarianMT models (first time)...")
                self.tokenizer_en_fr = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-fr')
                self.model_en_fr = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-fr').to(device)

                self.tokenizer_fr_en = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-fr-en')
                self.model_fr_en = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-fr-en').to(device)

                # Save to cache
                print("   � Caching MarianMT models...")
                os.makedirs(en_fr_cache, exist_ok=True)
                os.makedirs(fr_en_cache, exist_ok=True)

                self.tokenizer_en_fr.save_pretrained(en_fr_cache)
                self.model_en_fr.save_pretrained(en_fr_cache)

                self.tokenizer_fr_en.save_pretrained(fr_en_cache)
                self.model_fr_en.save_pretrained(fr_en_cache)
                print(f"   ✅ Cached at: {cache_dir}")
        else:
            # No cache directory - download fresh
            print("   📥 Downloading MarianMT models...")
            self.tokenizer_en_fr = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-fr')
            self.model_en_fr = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-fr').to(device)

            self.tokenizer_fr_en = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-fr-en')
            self.model_fr_en = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-fr-en').to(device)

        print("✅ BlindMutationEngine initialized!")

    def mutate(self, text: str) -> Tuple[str, str]:
        """
        Apply ONE random blind mutation.

        Returns:
            (mutated_text, mutation_type)
        """
        mutation_type = random.choice([
            'synonym_replacement',
            'sentence_shuffle',
            'back_translate',
            'word_deletion',
            'character_noise'
        ])

        mutated_text = self._apply_mutation(text, mutation_type)
        return mutated_text, mutation_type

    def _apply_mutation(self, text: str, mutation_type: str) -> str:
        """Apply specific mutation type."""

        if mutation_type == 'synonym_replacement':
            return self._synonym_replacement(text)
        elif mutation_type == 'sentence_shuffle':
            return self._sentence_shuffle(text)
        elif mutation_type == 'back_translate':
            return self._back_translate(text)
        elif mutation_type == 'word_deletion':
            return self._word_deletion(text)
        elif mutation_type == 'character_noise':
            return self._character_noise(text)
        else:
            return text

    def _synonym_replacement(self, text: str, num_replacements: int = 3) -> str:
        """Replace 3-5 random words with Word2Vec most_similar synonyms."""
        words = text.split()
        if len(words) < 5:
            return text

        # Get named entities to avoid replacing them
        doc = self.nlp(text)
        entity_words = set([token.text.lower() for ent in doc.ents for token in ent])

        replaceable_indices = [
            i for i, word in enumerate(words)
            if word.lower() not in entity_words and word.isalpha() and len(word) > 3
        ]

        if not replaceable_indices:
            return text

        num_to_replace = min(num_replacements, len(replaceable_indices))
        indices_to_replace = random.sample(replaceable_indices, num_to_replace)

        for idx in indices_to_replace:
            word = words[idx]
            try:
                # Get most similar words from Word2Vec (semantic similarity only)
                similar_words = self.word2vec.most_similar(word, topn=5)
                # Pick random synonym (not necessarily more formal/archaic)
                synonym = random.choice([w for w, _ in similar_words])

                # Preserve capitalization
                if word[0].isupper():
                    synonym = synonym.capitalize()

                words[idx] = synonym
            except KeyError:
                # Word not in vocabulary - skip
                pass

        return ' '.join(words)

    def _sentence_shuffle(self, text: str) -> str:
        """Randomly shuffle sentences."""
        # Split into sentences (simple split on .!?)
        sentences = re.split(r'([.!?]+)', text)

        # Combine sentences with their punctuation
        sentence_pairs = []
        for i in range(0, len(sentences) - 1, 2):
            if i + 1 < len(sentences):
                sentence_pairs.append(sentences[i] + sentences[i + 1])
            else:
                sentence_pairs.append(sentences[i])

        if len(sentence_pairs) <= 1:
            return text

        # Shuffle randomly (NO logic about chronology/flow)
        random.shuffle(sentence_pairs)

        return ' '.join(sentence_pairs).strip()

    def _back_translate(self, text: str) -> str:
        """
        English → French → English back-translation.
        Translation artifacts emerge naturally (not directed toward any style).
        """
        try:
            # English → French
            inputs_en = self.tokenizer_en_fr(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs_en = {k: v.to(self.device) for k, v in inputs_en.items()}

            translated_fr = self.model_en_fr.generate(**inputs_en, max_length=512)
            french_text = self.tokenizer_en_fr.batch_decode(translated_fr, skip_special_tokens=True)[0]

            # French → English
            inputs_fr = self.tokenizer_fr_en(french_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs_fr = {k: v.to(self.device) for k, v in inputs_fr.items()}

            translated_en = self.model_fr_en.generate(**inputs_fr, max_length=512)
            back_translated = self.tokenizer_fr_en.batch_decode(translated_en, skip_special_tokens=True)[0]

            return back_translated
        except Exception as e:
            print(f"   ⚠️ Back-translation failed: {e}")
            return text

    def _word_deletion(self, text: str, deletion_rate: float = 0.10) -> str:
        """
        Delete 10% of words randomly.
        Exclude named entities (spaCy NER).
        NO awareness of removing modern tells - just noise injection.
        """
        doc = self.nlp(text)
        entity_indices = set()

        for ent in doc.ents:
            for token in ent:
                entity_indices.add(token.i)

        words = []
        for i, token in enumerate(doc):
            if i not in entity_indices and random.random() > deletion_rate:
                words.append(token.text_with_ws)
            elif i in entity_indices:
                words.append(token.text_with_ws)

        return ''.join(words).strip()

    def _character_noise(self, text: str, num_typos: int = 2) -> str:
        """
        Introduce 1-2 character swaps/deletions per paragraph.
        Random noise, not strategic misspellings.
        """
        text_list = list(text)
        if len(text_list) < 20:
            return text

        # Get indices of alphabetic characters (avoid punctuation)
        alpha_indices = [i for i, c in enumerate(text_list) if c.isalpha()]

        if not alpha_indices:
            return text

        num_to_modify = min(num_typos, len(alpha_indices))
        indices_to_modify = random.sample(alpha_indices, num_to_modify)

        for idx in indices_to_modify:
            mutation_choice = random.choice(['swap', 'delete'])

            if mutation_choice == 'swap' and idx + 1 < len(text_list):
                # Swap adjacent characters
                text_list[idx], text_list[idx + 1] = text_list[idx + 1], text_list[idx]
            elif mutation_choice == 'delete':
                # Delete character
                text_list[idx] = ''

        return ''.join(text_list)


In [ ]:

# Initialize blind mutation engine (will be used if running Condition A)
print("\n🔬 Initializing Blind Mutation Engine...")
try:
    blind_engine = BlindMutationEngine(word2vec_model, device=device, cache_dir=CACHE_DIR)
    print("✅ Blind Mutation Engine ready!")
except Exception as e:
    print(f"⚠️ Could not initialize Blind Engine: {e}")
    print("   Run dependency installation cell first.")
    blind_engine = None

In [ ]:
print("\n" + "="*80)
print("SCIENTIFIC INTERPRETATION")
print("="*80)

success = final_best['human_prob'] >= TARGET_FITNESS
improvement = final_best['human_prob'] - initial_results[0]['human_prob']

if success:
    print("\n" + "🎉"*40)
    print("🎉 GA SUCCESSFULLY EVOLVED ADVERSARIAL TEXT")
    print("🎉"*40)

    print(f"\n📝 Final Evolved Paragraph:")
    print("-"*80)
    print(final_best['text'])
    print("-"*80)

    print(f"\n🔬 INTERPRETATION:")
    print("   ➤ Detector is VULNERABLE to evolutionary attacks")
    print("   ➤ GA discovered Victorian markers through iterative mutation")
    print("   ➤ Evolutionary search can systematically probe detector weaknesses")
    print("   ➤ This demonstrates the need for adversarial training")

    print(f"\n💡 IMPLICATIONS:")
    print("   • Detector NOT production-ready without adversarial hardening")
    print("   • Evolutionary algorithms can discover bypass strategies")
    print("   • Victorian patterns are learnable through blind optimization")
    print("   • Recommend: Adversarial training with GA-evolved examples")

else:
    print("\n" + "✅"*40)
    print("✅ DETECTOR ROBUST: GA FAILED TO REACH TARGET")
    print("✅"*40)

    if improvement > 0.20:
        print(f"\n   Note: Significant improvement (+{improvement*100:.1f}%) but below target")
        print(f"         Detector is MODERATELY ROBUST")
    elif improvement > 0.10:
        print(f"\n   Note: Modest improvement (+{improvement*100:.1f}%)")
        print(f"         Detector is ROBUST")
    else:
        print(f"\n   Note: Minimal improvement (+{improvement*100:.1f}%)")
        print(f"         Detector is HIGHLY ROBUST")

    print(f"\n📝 Best Evolved Paragraph (Still Detected as AI):")
    print("-"*80)
    print(final_best['text'])
    print("-"*80)

    print(f"\n🔬 INTERPRETATION:")
    print("   ➤ Detector is ROBUST against evolutionary attacks")
    print("   ➤ Victorian patterns cannot be discovered through blind mutation")
    print("   ➤ Deep structural features (e.g., mean_drift) are unfakeable")
    print("   ➤ Guided mutations insufficient to fool detector")

    print(f"\n💡 IMPLICATIONS:")
    print("   • Detector learned deep patterns, not superficial markers")
    print("   • Evolutionary search cannot replicate Victorian authenticity")
    print("   • Domain-specific training (Victorian corpus) provides robustness")
    print("   • Detector MAY be suitable for production deployment")
    print("   • Still recommend red-team testing with human adversaries")

print("\n📊 Most Effective Mutation Strategies:")
top_3 = mutation_stats_df.head(3)
for i, (_, row) in enumerate(top_3.iterrows(), 1):
    print(f"   #{i}: {row['strategy']:25s} "
          f"({row['success_rate']*100:5.1f}% success, "
          f"avg Δ={row['avg_improvement']:+.4f})")

print(f"\n{'='*80}")
print("🎓 CONCLUSION: Both success and failure are scientifically valuable!")
print("   This experiment provides actionable insights for detector improvement.")
print("="*80)


# 🔍 DIAGNOSTIC: Why Is Fitness So Low?

This cell can be run to diagnose the issue if GA is stuck at 0-1% confidence

In [ ]:


print("=" * 80)
print("DIAGNOSTIC: ANALYZING LOW FITNESS PROBLEM")
print("=" * 80)

# Check if we have results
if 'initial_results' not in globals() or len(initial_results) == 0:
    print("\n❌ No results found. Run the initial population generation first.")
else:
    print(f"\n📊 INITIAL POPULATION FITNESS ANALYSIS:")
    print(f"   Best:  {initial_results[0]['human_prob']*100:.4f}% Human")
    print(f"   Worst: {initial_results[-1]['human_prob']*100:.4f}% Human")
    print(f"   Average: {np.mean([r['human_prob'] for r in initial_results])*100:.4f}% Human")

    # Sample text analysis
    print(f"\n📝 SAMPLE INITIAL TEXT (Best Individual):")
    print("-" * 80)
    best_text = initial_results[0]['text']
    print(best_text[:300] + ("..." if len(best_text) > 300 else ""))
    print("-" * 80)

    # Victorian marker analysis
    markers = count_victorian_markers(best_text)
    print(f"\n🔍 VICTORIAN MARKERS IN BEST INITIAL TEXT:")
    print(f"   Archaic conjunctions (ere, lest): {markers['archaic_conj']}")
    print(f"   Past tense markers (was, had):    {markers['past_tense']}")
    print(f"   First-person pronouns (I, he):    {markers['first_person']}")
    print(f"   Modern transitions (however):     {markers['modern_trans']}")
    print(f"   Victorian vocabulary:              {markers['victorian_vocab']}")

    # Compare to authentic Victorian
    print(f"\n📚 COMPARISON TO AUTHENTIC VICTORIAN TEXT:")
    print(f"   Authentic Victorian typically has:")
    print(f"      - Archaic conjunctions: 2-5 per paragraph")
    print(f"      - Past tense markers: 8-15 per paragraph")
    print(f"      - First-person pronouns: 3-8 per paragraph")
    print(f"      - Modern transitions: 0-1 (avoid 'however')")
    print(f"      - Victorian vocabulary: 3-7 rare words")

    # Diagnosis
    print(f"\n🩺 DIAGNOSIS:")

    if initial_results[0]['human_prob'] < 0.05:  # Less than 5%
        print(f"   ❌ CRITICAL: Initial fitness is EXTREMELY low (<5% Human)")
        print(f"   📌 PROBLEM: Generated text has strong AI signatures:")
        print(f"      - Likely uses present tense heavily")
        print(f"      - Likely uses 'however', 'therefore', 'additionally'")
        print(f"      - Lacks archaic vocabulary and conjunctions")
        print(f"      - Uses impersonal third-person perspective")

        print(f"\n💡 SOLUTIONS:")
        print(f"   1. Use STRONGER mutation prompts (add more explicit Victorian hints)")
        print(f"   2. Increase mutation intensity (temperature, more aggressive rewrites)")
        print(f"   3. Add a 'Victorian style injection' mutation strategy")
        print(f"   4. Consider hybrid approach: naive initial + strong mutations")

    elif initial_results[0]['human_prob'] < 0.15:  # Less than 15%
        print(f"   ⚠️  WARNING: Initial fitness is very low (<15% Human)")
        print(f"   📌 PROBLEM: Text is detectably AI but mutations may help")
        print(f"      - Current mutations may be too subtle")
        print(f"      - Need more aggressive Victorian pattern injection")

        print(f"\n💡 SOLUTIONS:")
        print(f"   1. Increase mutation strength")
        print(f"   2. Add Victorian-specific mutation strategy")
        print(f"   3. Continue GA but expect slow progress")

    else:
        print(f"   ✓ Initial fitness is reasonable (>15% Human)")
        print(f"   📌 GA should be able to improve with current mutations")

    # Check if mutations are helping
    if 'history' in globals() and len(history) > 1:
        print(f"\n📈 MUTATION EFFECTIVENESS CHECK:")
        gen1_fitness = history[0]['best_fitness']
        gen_last_fitness = history[-1]['best_fitness']
        improvement = (gen_last_fitness - gen1_fitness) * 100

        print(f"   Generation 1:  {history[0]['best_human_prob']*100:.2f}% Human")
        print(f"   Generation {len(history)}: {history[-1]['best_human_prob']*100:.2f}% Human")
        print(f"   Improvement:   {improvement:+.4f}%")

        if abs(improvement) < 0.01:  # Less than 0.01% improvement
            print(f"\n   ❌ PROBLEM: Mutations are NOT improving fitness!")
            print(f"   💡 Recommendation: Add stronger mutation strategy (see below)")

print(f"\n{'='*80}")



---

# EXPERIMENT: Add Stronger Mutation Strategy

If mutations aren't working, add this **Victorian Pattern Injection** strategy.
# STRONGER MUTATION STRATEGY (if current ones aren't working)
# This provides MORE guidance while still requiring discovery


In [ ]:

ENHANCED_MUTATION_STRATEGIES = [
    # Original 8 strategies (keep these)
    *MUTATION_STRATEGIES,

    # NEW: Victorian Pattern Injection (stronger guidance)
    {
        'name': 'victorian_pattern_injection',
        'prompt': """Rewrite this paragraph in the style of classic 19th-century British literature
(like Arthur Conan Doyle or Robert Louis Stevenson). Use:
- More formal, antiquated language
- Past tense narration
- First-person or close third-person perspective
- Complex sentence structures with semicolons
- Rich, atmospheric description
- Avoid modern phrases like 'however' or 'therefore' as transitions

Original paragraph:
{text}

Rewritten in Victorian style (100-150 words):""",
        'description': 'Strong Victorian style injection (more explicit guidance)'
    },

    # NEW: Archaic Language Boost
    {
        'name': 'archaic_language_boost',
        'prompt': """Rewrite this paragraph using older, more archaic English. Replace modern
words with their older equivalents. Use formal conjunctions and connectives
that sound like they're from the 1800s. Make it sound like it was written
over 100 years ago.

Original paragraph:
{text}

Rewritten with archaic language (100-150 words):""",
        'description': 'Injects archaic vocabulary and phrasing'
    },

    # NEW: Atmospheric Storytelling
    {
        'name': 'atmospheric_storytelling',
        'prompt': """Rewrite this paragraph as if it's from a Gothic mystery novel from the 1800s.
Add atmospheric details, ominous descriptions, and a narrative voice that
draws the reader into a dark, mysterious world. Use rich, evocative language.

Original paragraph:
{text}

Rewritten as Gothic atmospheric narrative (100-150 words):""",
        'description': 'Gothic Victorian narrative style'
    }
]

print("💪 ENHANCED MUTATION STRATEGIES DEFINED")
print("=" * 80)
print(f"Total strategies: {len(ENHANCED_MUTATION_STRATEGIES)}")
print(f"  - Original:  {len(MUTATION_STRATEGIES)} (subtle guidance)")
print(f"  - Enhanced:  {len(ENHANCED_MUTATION_STRATEGIES) - len(MUTATION_STRATEGIES)} (stronger guidance)")
print("\n⚠️  These provide MORE explicit Victorian hints while still requiring GA to discover")
print("   which specific patterns work best (ere vs. lest, past tense density, etc.)")
print("\n📝 To use: Replace MUTATION_STRATEGIES with ENHANCED_MUTATION_STRATEGIES")
print("   in the mutation selector initialization.")
print("=" * 80)


# 🔄 RE-RUN GA with Enhanced Mutations

If you want to restart the GA with stronger mutations, run this cell.
# RESTART GA WITH ENHANCED MUTATIONS


In [ ]:


print("=" * 80)
print("🔄 RESTARTING GA WITH ENHANCED MUTATION STRATEGIES")
print("=" * 80)

# Use enhanced strategies
mutation_selector_enhanced = AdaptiveMutationSelector(ENHANCED_MUTATION_STRATEGIES)

# Keep the same initial population (don't regenerate)
if 'initial_results' not in globals():
    print("❌ ERROR: No initial population found!")
    print("   Run the initial population generation first.")
else:
    population_enhanced = [r['text'] for r in initial_results]
    history_enhanced = []
    start_time_enhanced = datetime.now()

    print(f"\n✅ Using existing initial population ({len(population_enhanced)} individuals)")
    print(f"   Initial best: {initial_results[0]['human_prob']*100:.2f}% Human")
    print(f"\n🚀 Starting evolution with {len(ENHANCED_MUTATION_STRATEGIES)} mutation strategies...")
    print(f"   ({len(ENHANCED_MUTATION_STRATEGIES) - len(MUTATION_STRATEGIES)} new strategies added)")
    print("=" * 80)

    for generation in range(1, NUM_GENERATIONS + 1):
        print(f"\n{'='*80}")
        print(f"GENERATION {generation}/{NUM_GENERATIONS} (ENHANCED)")
        print(f"{'='*80}")

        # Evaluate population
        evaluated_enhanced = []
        for text in population_enhanced:
            result = calculate_comprehensive_fitness(text, distilbert_predictor)
            result['text'] = text
            evaluated_enhanced.append(result)

        evaluated_enhanced = sorted(evaluated_enhanced, key=lambda x: x['fitness'], reverse=True)

        # Track history
        best = evaluated_enhanced[0]
        avg_fitness = np.mean([e['fitness'] for e in evaluated_enhanced])

        history_enhanced.append({
            'generation': generation,
            'best_fitness': best['fitness'],
            'best_human_prob': best['human_prob'],
            'avg_fitness': avg_fitness,
            'best_text': best['text'],
            'best_predicted_class': best['predicted_class']
        })

        # Print statistics
        improvement = (best['human_prob'] - initial_results[0]['human_prob']) * 100

        print(f"\n📊 Generation {generation} Statistics:")
        print(f"   Best:    {best['fitness']:.4f} ({best['human_prob']*100:.2f}% Human, {best['predicted_class']})")
        print(f"   Average: {avg_fitness:.4f}")
        print(f"   Improvement from initial: {improvement:+.2f}%")

        # Check if target achieved
        if best['human_prob'] >= TARGET_FITNESS:
            print(f"\n{'🎉'*40}")
            print(f"🎉 SUCCESS! Achieved >{TARGET_FITNESS*100:.0f}% Human confidence!")
            print(f"{'🎉'*40}")
            break

        # Selection
        survivors = [e['text'] for e in evaluated_enhanced[:TOP_K_SELECTION]]
        survivor_fitness = [e['fitness'] for e in evaluated_enhanced[:TOP_K_SELECTION]]

        print(f"\n🔍 Top {TOP_K_SELECTION} Survivors:")
        for i, e in enumerate(evaluated_enhanced[:TOP_K_SELECTION], 1):
            print(f"   #{i}: {e['fitness']:.4f} ({e['human_prob']*100:.2f}% Human)")

        if generation == NUM_GENERATIONS:
            break

        # Mutation
        print(f"\n🧬 Generating Generation {generation + 1}...")
        next_population = survivors.copy()

        mutations_needed = POPULATION_SIZE - TOP_K_SELECTION
        mutations_per_parent = max(1, mutations_needed // TOP_K_SELECTION)

        mutation_count = 0
        for parent_idx, (parent_text, parent_fitness) in enumerate(zip(survivors, survivor_fitness), 1):
            for mutation_idx in range(mutations_per_parent):
                if len(next_population) >= POPULATION_SIZE:
                    break

                # Select strategy (adaptive - will favor Victorian injection if it works)
                strategy = mutation_selector_enhanced.select_strategy(generation)

                try:
                    prompt = strategy['prompt'].format(text=parent_text)

                    data = {
                        "contents": [{"parts": [{"text": prompt}]}],
                        "safetySettings": [
                            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
                        ],
                        "generationConfig": {"temperature": 0.7, "maxOutputTokens": 8192}
                    }

                    response = requests.post(URL, headers=HEADERS, data=json.dumps(data), timeout=30)

                    if response.status_code == 200:
                        result = response.json()
                        mutated_text = result['candidates'][0]['content']['parts'][0]['text']
                    else:
                        mutated_text = parent_text

                    # Evaluate
                    child_result = calculate_comprehensive_fitness(mutated_text, distilbert_predictor)

                    # Record
                    mutation_selector_enhanced.record_mutation(strategy['name'], parent_fitness, child_result['fitness'])

                    next_population.append(mutated_text)
                    mutation_count += 1

                    symbol = '✓' if child_result['fitness'] > parent_fitness else '✗'
                    print(f"   [{mutation_count}/{mutations_needed}] P{parent_idx}, M{mutation_idx+1} "
                          f"({strategy['name'][:20]}): {child_result['human_prob']*100:.1f}% Human {symbol}")

                    time.sleep(2.0)

                except Exception as e:
                    print(f"   [{mutation_count+1}] Error: {e}")
                    next_population.append(parent_text)
                    time.sleep(2.0)

        while len(next_population) < POPULATION_SIZE:
            next_population.append(random.choice(survivors))

        population_enhanced = next_population[:POPULATION_SIZE]

    end_time_enhanced = datetime.now()
    elapsed_enhanced = end_time_enhanced - start_time_enhanced

    print(f"\n{'='*80}")
    print(f"⏱️  Enhanced GA Runtime: {elapsed_enhanced}")
    print(f"{'='*80}")

    # Save enhanced results
    history_enhanced_df = pd.DataFrame(history_enhanced)
    history_enhanced_df.to_csv(f"{OUTPUT_DIR}/evolution_history_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/evolution_history_enhanced.csv")

    mutation_selector_enhanced.print_statistics()

In [ ]:
print("=" * 80)
print("💾 SAVING COMPLETE ENHANCED GA RESULTS")
print("=" * 80)

if 'history_enhanced' not in globals() or len(history_enhanced) == 0:
    print("\n❌ No enhanced results found. Run the enhanced GA first.")
else:
    # Get final best text
    final_evaluated_enhanced = []
    for text in population_enhanced:
        result = calculate_comprehensive_fitness(text, distilbert_predictor)
        result['text'] = text
        final_evaluated_enhanced.append(result)

    final_evaluated_enhanced = sorted(final_evaluated_enhanced, key=lambda x: x['fitness'], reverse=True)
    final_best_enhanced = final_evaluated_enhanced[0]

    # Create comprehensive report
    print("\n📊 ENHANCED GA FINAL STATISTICS:")
    print("=" * 80)
    print(f"Success: {'✅ YES' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else '❌ NO'}")
    print(f"Initial best:  {initial_results[0]['human_prob']*100:.2f}% Human")
    print(f"Final best:    {final_best_enhanced['human_prob']*100:.2f}% Human")
    print(f"Improvement:   {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%")
    print(f"Generations:   {len(history_enhanced)}")
    print(f"Runtime:       {elapsed_enhanced}")
    print(f"Target:        {TARGET_FITNESS*100:.0f}% Human")
    print("=" * 80)

    # 1. Save evolution history (already saved, but add more detail)
    history_enhanced_detailed = pd.DataFrame(history_enhanced)
    history_enhanced_detailed.to_csv(f"{OUTPUT_DIR}/evolution_history_enhanced.csv", index=False)
    print(f"\n✅ Saved: {OUTPUT_DIR}/evolution_history_enhanced.csv")

    # 2. Save mutation strategy effectiveness
    mutation_stats_enhanced = mutation_selector_enhanced.get_statistics()
    mutation_stats_enhanced.to_csv(f"{OUTPUT_DIR}/mutation_strategy_effectiveness_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/mutation_strategy_effectiveness_enhanced.csv")

    # 3. Save Victorian marker evolution
    markers_df_enhanced = analyze_victorian_markers_evolution(history_enhanced)
    markers_df_enhanced.to_csv(f"{OUTPUT_DIR}/victorian_markers_evolution_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/victorian_markers_evolution_enhanced.csv")

    # 4. Save final population
    final_population_enhanced = pd.DataFrame([{
        'rank': i,
        'text': e['text'],
        'fitness': e['fitness'],
        'human_prob': e['human_prob'],
        'ai_prob': e['ai_prob'],
        'predicted_class': e['predicted_class'],
        'word_count': e['word_count']
    } for i, e in enumerate(final_evaluated_enhanced, 1)])
    final_population_enhanced.to_csv(f"{OUTPUT_DIR}/final_population_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/final_population_enhanced.csv")

    # 5. Save best evolved text with full analysis
    with open(f"{OUTPUT_DIR}/best_evolved_text_enhanced.txt", 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("BEST EVOLVED TEXT (ENHANCED GA)\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"SUCCESS: {'YES - Achieved target!' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else 'NO - Did not reach target'}\n")
        f.write(f"Fitness: {final_best_enhanced['fitness']:.4f}\n")
        f.write(f"Human Probability: {final_best_enhanced['human_prob']:.4f} ({final_best_enhanced['human_prob']*100:.2f}%)\n")
        f.write(f"AI Probability: {final_best_enhanced['ai_prob']:.4f} ({final_best_enhanced['ai_prob']*100:.2f}%)\n")
        f.write(f"Predicted Class: {final_best_enhanced['predicted_class']}\n")
        f.write(f"Word Count: {final_best_enhanced['word_count']}\n")
        f.write(f"Generations: {len(history_enhanced)}\n")
        f.write(f"Runtime: {elapsed_enhanced}\n")
        f.write(f"Initial fitness: {initial_results[0]['human_prob']*100:.2f}% Human\n")
        f.write(f"Improvement: {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%\n")
        f.write("\n" + "=" * 80 + "\n")
        f.write("EVOLVED TEXT:\n")
        f.write("=" * 80 + "\n\n")
        f.write(final_best_enhanced['text'])
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("VICTORIAN MARKER ANALYSIS:\n")
        f.write("=" * 80 + "\n\n")

        # Victorian markers
        final_markers = count_victorian_markers(final_best_enhanced['text'])
        initial_markers = count_victorian_markers(initial_results[0]['text'])

        f.write(f"Archaic conjunctions (ere, lest):  Initial: {initial_markers['archaic_conj']:2d}  →  Final: {final_markers['archaic_conj']:2d}  ({final_markers['archaic_conj'] - initial_markers['archaic_conj']:+d})\n")
        f.write(f"Past tense markers (was, had):     Initial: {initial_markers['past_tense']:2d}  →  Final: {final_markers['past_tense']:2d}  ({final_markers['past_tense'] - initial_markers['past_tense']:+d})\n")
        f.write(f"First-person pronouns (I, he):     Initial: {initial_markers['first_person']:2d}  →  Final: {final_markers['first_person']:2d}  ({final_markers['first_person'] - initial_markers['first_person']:+d})\n")
        f.write(f"Modern transitions (however):      Initial: {initial_markers['modern_trans']:2d}  →  Final: {final_markers['modern_trans']:2d}  ({final_markers['modern_trans'] - initial_markers['modern_trans']:+d})\n")
        f.write(f"Victorian vocabulary:               Initial: {initial_markers['victorian_vocab']:2d}  →  Final: {final_markers['victorian_vocab']:2d}  ({final_markers['victorian_vocab'] - initial_markers['victorian_vocab']:+d})\n")
        f.write(f"Semicolons:                         Initial: {initial_markers['semicolons']:2d}  →  Final: {final_markers['semicolons']:2d}  ({final_markers['semicolons'] - initial_markers['semicolons']:+d})\n")
        f.write(f"Em-dashes:                          Initial: {initial_markers['em_dashes']:2d}  →  Final: {final_markers['em_dashes']:2d}  ({final_markers['em_dashes'] - initial_markers['em_dashes']:+d})\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("MOST EFFECTIVE MUTATION STRATEGIES:\n")
        f.write("=" * 80 + "\n\n")

        for i, (_, row) in enumerate(mutation_stats_enhanced.head(5).iterrows(), 1):
            f.write(f"{i}. {row['strategy']:30s} - {row['success_rate']*100:5.1f}% success ({row['attempts']:.0f} attempts, avg Δ={row['avg_improvement']:+.4f})\n")

        f.write("\n" + "=" * 80 + "\n")

    print(f"✅ Saved: {OUTPUT_DIR}/best_evolved_text_enhanced.txt")

    # 6. Save comparison: Original GA vs Enhanced GA
    comparison_data = {
        'Metric': [
            'Initial Best Fitness',
            'Final Best Fitness',
            'Improvement',
            'Generations Run',
            'Target Achieved',
            'Runtime',
            'Best Strategy (Original)',
            'Best Strategy (Enhanced)'
        ],
        'Original GA': [
            f"{initial_results[0]['human_prob']*100:.2f}% Human" if 'history' in globals() and len(history) > 0 else 'N/A',
            f"{history[-1]['best_human_prob']*100:.2f}% Human" if 'history' in globals() and len(history) > 0 else 'N/A',
            f"{(history[-1]['best_human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%" if 'history' in globals() and len(history) > 0 else 'N/A',
            f"{len(history)}" if 'history' in globals() else 'N/A',
            'NO' if 'history' not in globals() or len(history) == 0 or history[-1]['best_human_prob'] < TARGET_FITNESS else 'YES',
            str(elapsed) if 'elapsed' in globals() else 'N/A',
            mutation_selector.get_statistics().iloc[0]['strategy'] if 'mutation_selector' in globals() else 'N/A',
            ''
        ],
        'Enhanced GA': [
            f"{initial_results[0]['human_prob']*100:.2f}% Human",
            f"{final_best_enhanced['human_prob']*100:.2f}% Human",
            f"{(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%",
            f"{len(history_enhanced)}",
            'YES' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else 'NO',
            str(elapsed_enhanced),
            '',
            mutation_stats_enhanced.iloc[0]['strategy']
        ]
    }

    comparison_df = pd.DataFrame(comparison_data)
    comparison_df.to_csv(f"{OUTPUT_DIR}/original_vs_enhanced_comparison.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/original_vs_enhanced_comparison.csv")

    # 7. Create visualization of marker evolution
    print(f"\n📊 Creating Victorian marker evolution visualization...")
    plot_victorian_markers_evolution(markers_df_enhanced, f"{OUTPUT_DIR}/victorian_markers_evolution_enhanced.png")

    # 8. Save complete summary report
    with open(f"{OUTPUT_DIR}/EXPERIMENT_SUMMARY_ENHANCED.txt", 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("TASK 4: GENETIC ALGORITHM v2.0 - ENHANCED EXPERIMENT SUMMARY\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Experiment: Enhanced GA with Victorian Pattern Injection\n\n")

        f.write("=" * 80 + "\n")
        f.write("CONFIGURATION\n")
        f.write("=" * 80 + "\n")
        f.write(f"Population Size: {POPULATION_SIZE}\n")
        f.write(f"Generations: {NUM_GENERATIONS}\n")
        f.write(f"Selection: Top {TOP_K_SELECTION}\n")
        f.write(f"Target Fitness: >{TARGET_FITNESS*100:.0f}% Human confidence\n")
        f.write(f"Mutation Strategies: {len(ENHANCED_MUTATION_STRATEGIES)} (8 original + 3 enhanced)\n")
        f.write(f"Model: DistilBERT-LoRA (99.71% accuracy)\n\n")

        f.write("=" * 80 + "\n")
        f.write("RESULTS\n")
        f.write("=" * 80 + "\n")
        f.write(f"SUCCESS: {'✅ YES - Target achieved!' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else '❌ NO - Target not reached'}\n\n")
        f.write(f"Initial Best:     {initial_results[0]['human_prob']*100:.2f}% Human (detector caught it)\n")
        f.write(f"Final Best:       {final_best_enhanced['human_prob']*100:.2f}% Human (detector {'fooled!' if final_best_enhanced['human_prob'] > 0.5 else 'caught it'})\n")
        f.write(f"Improvement:      {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%\n")
        f.write(f"Generations:      {len(history_enhanced)}/{NUM_GENERATIONS}\n")
        f.write(f"Runtime:          {elapsed_enhanced}\n\n")

        f.write("=" * 80 + "\n")
        f.write("MUTATION STRATEGY EFFECTIVENESS\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"{'Rank':<6}{'Strategy':<35}{'Success Rate':<15}{'Attempts':<12}{'Avg Δ':<12}\n")
        f.write("-" * 80 + "\n")
        for i, (_, row) in enumerate(mutation_stats_enhanced.iterrows(), 1):
            f.write(f"{i:<6}{row['strategy']:<35}{row['success_rate']*100:>6.1f}%       {row['attempts']:>6.0f}      {row['avg_improvement']:>+8.4f}\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("VICTORIAN MARKER EVOLUTION\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"{'Marker':<30}{'Initial':<10}{'Final':<10}{'Change':<10}\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Archaic conjunctions':<30}{initial_markers['archaic_conj']:<10}{final_markers['archaic_conj']:<10}{final_markers['archaic_conj'] - initial_markers['archaic_conj']:+d}\n")
        f.write(f"{'Past tense markers':<30}{initial_markers['past_tense']:<10}{final_markers['past_tense']:<10}{final_markers['past_tense'] - initial_markers['past_tense']:+d}\n")
        f.write(f"{'First-person pronouns':<30}{initial_markers['first_person']:<10}{final_markers['first_person']:<10}{final_markers['first_person'] - initial_markers['first_person']:+d}\n")
        f.write(f"{'Modern transitions':<30}{initial_markers['modern_trans']:<10}{final_markers['modern_trans']:<10}{final_markers['modern_trans'] - initial_markers['modern_trans']:+d}\n")
        f.write(f"{'Victorian vocabulary':<30}{initial_markers['victorian_vocab']:<10}{final_markers['victorian_vocab']:<10}{final_markers['victorian_vocab'] - initial_markers['victorian_vocab']:+d}\n")
        f.write(f"{'Semicolons':<30}{initial_markers['semicolons']:<10}{final_markers['semicolons']:<10}{final_markers['semicolons'] - initial_markers['semicolons']:+d}\n")
        f.write(f"{'Em-dashes':<30}{initial_markers['em_dashes']:<10}{final_markers['em_dashes']:<10}{final_markers['em_dashes'] - initial_markers['em_dashes']:+d}\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("SCIENTIFIC INTERPRETATION\n")
        f.write("=" * 80 + "\n\n")

        if final_best_enhanced['human_prob'] >= TARGET_FITNESS:
            f.write("✅ DETECTOR IS VULNERABLE TO EVOLUTIONARY ATTACKS\n\n")
            f.write("Key Findings:\n")
            f.write(f"  • GA successfully evolved text from {initial_results[0]['human_prob']*100:.2f}% to {final_best_enhanced['human_prob']*100:.2f}% Human\n")
            f.write(f"  • Victorian markers emerged through guided mutation\n")
            f.write(f"  • Most effective strategy: {mutation_stats_enhanced.iloc[0]['strategy']} ({mutation_stats_enhanced.iloc[0]['success_rate']*100:.1f}% success)\n")
            f.write(f"  • Evolution discovered that {mutation_stats_enhanced.iloc[0]['strategy']} patterns fool detector\n\n")
            f.write("Implications:\n")
            f.write("  • Detector NOT production-ready without adversarial hardening\n")
            f.write("  • Evolutionary algorithms can systematically probe weaknesses\n")
            f.write("  • Victorian patterns are learnable through guided optimization\n")
            f.write("  • Recommend: Adversarial training with GA-evolved examples\n\n")
            f.write("Next Steps:\n")
            f.write("  1. Retrain detector with adversarial examples from this GA\n")
            f.write("  2. Implement adversarial training loop\n")
            f.write("  3. Test robustness with multiple GA runs\n")
            f.write("  4. Red-team testing with human adversaries\n")
        else:
            f.write("✅ DETECTOR IS ROBUST AGAINST EVOLUTIONARY ATTACKS\n\n")
            f.write("Key Findings:\n")
            f.write(f"  • GA improved fitness from {initial_results[0]['human_prob']*100:.2f}% to {final_best_enhanced['human_prob']*100:.2f}% Human\n")
            f.write(f"  • Improvement of {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}% but below {TARGET_FITNESS*100:.0f}% target\n")
            f.write(f"  • Even enhanced mutations could not fool detector\n")
            f.write(f"  • Deep structural features appear unfakeable\n\n")
            f.write("Implications:\n")
            f.write("  • Detector learned deep patterns beyond superficial markers\n")
            f.write("  • Victorian style cannot be easily replicated by AI\n")
            f.write("  • Domain-specific training provides robustness\n")
            f.write("  • Detector MAY be suitable for production deployment\n\n")
            f.write("Next Steps:\n")
            f.write("  1. Additional red-team testing recommended\n")
            f.write("  2. Test with human-crafted adversarial examples\n")
            f.write("  3. Validate on diverse Victorian authors\n")
            f.write("  4. Monitor for concept drift over time\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("FILES GENERATED\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"  • evolution_history_enhanced.csv\n")
        f.write(f"  • mutation_strategy_effectiveness_enhanced.csv\n")
        f.write(f"  • victorian_markers_evolution_enhanced.csv\n")
        f.write(f"  • final_population_enhanced.csv\n")
        f.write(f"  • best_evolved_text_enhanced.txt\n")
        f.write(f"  • original_vs_enhanced_comparison.csv\n")
        f.write(f"  • victorian_markers_evolution_enhanced.png\n")
        f.write(f"  • EXPERIMENT_SUMMARY_ENHANCED.txt (this file)\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")

    print(f"✅ Saved: {OUTPUT_DIR}/EXPERIMENT_SUMMARY_ENHANCED.txt")

    print("\n" + "=" * 80)
    print("✅ ALL ENHANCED RESULTS SAVED SUCCESSFULLY!")
    print("=" * 80)
    print(f"\n📁 Output directory: {OUTPUT_DIR}/")
    print("\n📊 Files created:")
    print("   1. evolution_history_enhanced.csv - Generation-by-generation fitness")
    print("   2. mutation_strategy_effectiveness_enhanced.csv - Which strategies worked")
    print("   3. victorian_markers_evolution_enhanced.csv - Marker emergence data")
    print("   4. final_population_enhanced.csv - All final individuals ranked")
    print("   5. best_evolved_text_enhanced.txt - Winner with full analysis")
    print("   6. original_vs_enhanced_comparison.csv - Original vs Enhanced GA")
    print("   7. victorian_markers_evolution_enhanced.png - Evolution visualization")
    print("   8. EXPERIMENT_SUMMARY_ENHANCED.txt - Complete summary report")
    print("\n🎉 Your enhanced GA achieved " +
          f"{final_best_enhanced['human_prob']*100:.2f}% Human confidence!")
    print(f"   Target was {TARGET_FITNESS*100:.0f}% - " +
          ("✅ SUCCESS!" if final_best_enhanced['human_prob'] >= TARGET_FITNESS else "❌ Not reached"))
    print("=" * 80)

---

# 📊 ABLATION STUDY ANALYSIS

Run these cells AFTER completing the ablation study to analyze results.

In [ ]:
# Analysis functions for ablation study
from scipy import stats

def plot_fitness_comparison(ablation_results: Dict[str, List[ExperimentResult]],
                            output_dir: str):
    """
    Create comprehensive fitness comparison plots.

    1. Line plot: Fitness evolution over generations (3 lines with mean ± std)
    2. Boxplot: Final fitness distributions (3 boxes)
    """

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Fitness evolution
    colors = {'blind': 'blue', 'guided': 'green', 'hybrid': 'purple'}

    for condition, results in ablation_results.items():
        if not results:
            continue

        # Aggregate across seeds
        max_gens = max(len(r.history) for r in results)
        fitness_by_gen = defaultdict(list)

        for result in results:
            for record in result.history:
                gen = record['generation']
                fitness_by_gen[gen].append(record['best_human_prob'])

        # Calculate mean and std
        generations = sorted(fitness_by_gen.keys())
        means = [np.mean(fitness_by_gen[g]) for g in generations]
        stds = [np.std(fitness_by_gen[g]) for g in generations]

        # Plot line with shaded region
        ax1.plot(generations, means, label=condition.upper(),
                color=colors[condition], linewidth=2.5, marker='o')
        ax1.fill_between(generations,
                        [m - s for m, s in zip(means, stds)],
                        [m + s for m, s in zip(means, stds)],
                        alpha=0.2, color=colors[condition])

    ax1.set_xlabel('Generation', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Best Human Probability', fontsize=12, fontweight='bold')
    ax1.set_title('Fitness Evolution Across Conditions', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=TARGET_FITNESS, color='red', linestyle='--',
                label=f'Target ({TARGET_FITNESS*100:.0f}%)', linewidth=2)

    # Plot 2: Final fitness distribution
    final_fitness_data = []
    labels = []

    for condition in ['blind', 'guided', 'hybrid']:
        if condition in ablation_results and ablation_results[condition]:
            final_probs = [r.best_individual['human_prob']
                          for r in ablation_results[condition]]
            final_fitness_data.append(final_probs)
            labels.append(condition.upper())

    bp = ax2.boxplot(final_fitness_data, labels=labels, patch_artist=True,
                     notch=True, widths=0.6)

    # Color boxes
    for patch, condition in zip(bp['boxes'], ['blind', 'guided', 'hybrid']):
        patch.set_facecolor(colors[condition])
        patch.set_alpha(0.6)

    ax2.set_ylabel('Final Human Probability', fontsize=12, fontweight='bold')
    ax2.set_title('Final Fitness Distribution (3 runs per condition)',
                 fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(y=TARGET_FITNESS, color='red', linestyle='--', linewidth=2)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/ablation_fitness_comparison.png", dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {output_dir}/ablation_fitness_comparison.png")
    plt.close()

def analyze_convergence(ablation_results: Dict[str, List[ExperimentResult]],
                       threshold: float = 0.70):
    """
    Calculate generations to reach threshold (e.g., 70% Human).

    Returns DataFrame with convergence statistics.
    """
    convergence_data = []

    for condition, results in ablation_results.items():
        for result in results:
            gens_to_converge = None

            for record in result.history:
                if record['best_human_prob'] >= threshold:
                    gens_to_converge = record['generation']
                    break

            convergence_data.append({
                'condition': condition,
                'seed': result.seed,
                'converged': gens_to_converge is not None,
                'generations_to_converge': gens_to_converge if gens_to_converge else 'DNF',
                'final_fitness': result.best_individual['human_prob']
            })

    df = pd.DataFrame(convergence_data)
    return df

def track_victorian_markers_all_conditions(ablation_results: Dict[str, List[ExperimentResult]],
                                          output_dir: str):
    """
    Track Victorian marker emergence for ALL conditions.
    Create 3 heatmaps (one per condition).
    """

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    for idx, condition in enumerate(['blind', 'guided', 'hybrid']):
        if condition not in ablation_results or not ablation_results[condition]:
            continue

        # Average across seeds
        all_markers = []

        for result in ablation_results[condition]:
            for record in result.history:
                markers = count_victorian_markers(record['best_text'])
                markers['generation'] = record['generation']
                all_markers.append(markers)

        if not all_markers:
            continue

        markers_df = pd.DataFrame(all_markers)

        # Group by generation and average
        markers_agg = markers_df.groupby('generation').mean()

        # Create heatmap
        marker_cols = ['archaic_conj', 'past_tense', 'first_person',
                      'modern_trans', 'victorian_vocab']
        heatmap_data = markers_agg[marker_cols].T

        sns.heatmap(heatmap_data, ax=axes[idx], cmap='YlOrRd',
                   annot=True, fmt='.1f', cbar_kws={'label': 'Count'})
        axes[idx].set_title(f'{condition.upper()} Condition',
                           fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Generation', fontsize=10)
        axes[idx].set_ylabel('Victorian Marker', fontsize=10)

    plt.suptitle('Victorian Marker Emergence Across All Conditions',
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ablation_victorian_markers_heatmap.png",
               dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {output_dir}/ablation_victorian_markers_heatmap.png")
    plt.close()

def statistical_comparison(ablation_results: Dict[str, List[ExperimentResult]]):
    """
    Perform statistical tests:
    - One-way ANOVA: Do final fitness means differ?
    - Tukey's HSD post-hoc: Which pairs differ significantly?
    """

    print("\n" + "="*80)
    print("STATISTICAL COMPARISON")
    print("="*80)

    # Extract final fitness for each condition
    groups = {}
    for condition, results in ablation_results.items():
        if results:
            groups[condition] = [r.best_individual['human_prob'] for r in results]

    if len(groups) < 2:
        print("⚠️ Need at least 2 conditions for statistical comparison")
        return

    # Print descriptive statistics
    print("\n📊 Descriptive Statistics:")
    print(f"{'Condition':<12} {'Mean':<10} {'Std Dev':<10} {'Min':<10} {'Max':<10}")
    print("-"*52)

    for condition, values in groups.items():
        print(f"{condition.upper():<12} {np.mean(values):<10.4f} "
              f"{np.std(values):<10.4f} {np.min(values):<10.4f} {np.max(values):<10.4f}")

    # One-way ANOVA
    print("\n📈 One-Way ANOVA:")
    print("   H₀: μ_blind = μ_guided = μ_hybrid (no difference)")
    print("   H₁: At least one mean differs")

    f_stat, p_value = stats.f_oneway(*groups.values())

    print(f"\n   F-statistic: {f_stat:.4f}")
    print(f"   p-value: {p_value:.4f}")

    if p_value < 0.05:
        print(f"   ✅ SIGNIFICANT DIFFERENCE (p < 0.05)")
        print(f"      At least one condition differs from others")
    else:
        print(f"   ❌ NO SIGNIFICANT DIFFERENCE (p ≥ 0.05)")
        print(f"      All conditions perform similarly")

    # Pairwise t-tests (post-hoc)
    print("\n📊 Pairwise Comparisons (t-tests):")
    print(f"{'Comparison':<25} {'t-statistic':<15} {'p-value':<12} {'Significant?':<15}")
    print("-"*67)

    conditions = list(groups.keys())
    for i in range(len(conditions)):
        for j in range(i + 1, len(conditions)):
            cond1, cond2 = conditions[i], conditions[j]
            t_stat, p_val = stats.ttest_ind(groups[cond1], groups[cond2])
            sig = "YES (p < 0.05)" if p_val < 0.05 else "NO (p ≥ 0.05)"
            print(f"{cond1.upper()} vs {cond2.upper():<15} {t_stat:<15.4f} "
                  f"{p_val:<12.4f} {sig:<15}")

    # Effect size (Cohen's d)
    print("\n📏 Effect Sizes (Cohen's d):")
    print(f"{'Comparison':<25} {'Cohen\\'s d':<15} {'Interpretation':<20}")
    print("-"*60)

    for i in range(len(conditions)):
        for j in range(i + 1, len(conditions)):
            cond1, cond2 = conditions[i], conditions[j]
            mean1, mean2 = np.mean(groups[cond1]), np.mean(groups[cond2])
            std1, std2 = np.std(groups[cond1], ddof=1), np.std(groups[cond2], ddof=1)
            pooled_std = np.sqrt((std1**2 + std2**2) / 2)
            cohens_d = (mean1 - mean2) / pooled_std if pooled_std > 0 else 0

            if abs(cohens_d) < 0.2:
                interpretation = "Negligible"
            elif abs(cohens_d) < 0.5:
                interpretation = "Small"
            elif abs(cohens_d) < 0.8:
                interpretation = "Medium"
            else:
                interpretation = "Large"

            print(f"{cond1.upper()} vs {cond2.upper():<15} {cohens_d:<15.4f} {interpretation:<20}")

    print("\n" + "="*80)

print("✅ Analysis functions defined!")

---

# 📋 Run Complete Analysis

Run this cell after ablation study completes to generate all reports.

In [ ]:
def generate_ablation_report(ablation_results: Dict[str, List[ExperimentResult]],
                            output_dir: str):
    """
    Generate complete ablation study report with all analysis.
    """

    print("\n" + "📊"*40)
    print("📊 GENERATING ABLATION STUDY REPORT")
    print("📊"*40)

    # Create output subdirectory
    ablation_dir = f"{output_dir}/ablation_study"
    os.makedirs(ablation_dir, exist_ok=True)

    # 1. Fitness comparison plots
    print("\n1️⃣ Creating fitness comparison plots...")
    plot_fitness_comparison(ablation_results, ablation_dir)

    # 2. Convergence analysis
    print("\n2️⃣ Analyzing convergence rates...")
    convergence_df = analyze_convergence(ablation_results, threshold=0.70)
    convergence_df.to_csv(f"{ablation_dir}/convergence_analysis.csv", index=False)
    print(f"✅ Saved: {ablation_dir}/convergence_analysis.csv")

    print("\nConvergence Summary:")
    print(convergence_df.groupby('condition')['converged'].sum())

    # 3. Victorian marker tracking
    print("\n3️⃣ Tracking Victorian marker emergence...")
    track_victorian_markers_all_conditions(ablation_results, ablation_dir)

    # 4. Statistical comparison
    print("\n4️⃣ Running statistical tests...")
    statistical_comparison(ablation_results)

    # 5. Save raw data
    print("\n5️⃣ Saving raw experimental data...")

    all_data = []
    for condition, results in ablation_results.items():
        for result in results:
            for record in result.history:
                all_data.append({
                    'condition': condition,
                    'seed': result.seed,
                    'generation': record['generation'],
                    'best_fitness': record['best_fitness'],
                    'best_human_prob': record['best_human_prob'],
                    'avg_fitness': record['avg_fitness']
                })

    all_data_df = pd.DataFrame(all_data)
    all_data_df.to_csv(f"{ablation_dir}/ablation_raw_data.csv", index=False)
    print(f"✅ Saved: {ablation_dir}/ablation_raw_data.csv")

    # 6. Save best individuals from each condition
    print("\n6️⃣ Saving best individuals...")

    with open(f"{ablation_dir}/best_individuals_all_conditions.txt", 'w') as f:
        f.write("="*80 + "\n")
        f.write("BEST EVOLVED INDIVIDUALS - ABLATION STUDY\n")
        f.write("="*80 + "\n\n")

        for condition, results in ablation_results.items():
            f.write(f"\n{'='*80}\n")
            f.write(f"CONDITION: {condition.upper()}\n")
            f.write(f"{'='*80}\n\n")

            for result in results:
                best = result.best_individual
                f.write(f"\n--- Seed {result.seed} ---\n")
                f.write(f"Human Probability: {best['human_prob']*100:.2f}%\n")
                f.write(f"Fitness: {best['fitness']:.4f}\n")
                f.write(f"Predicted: {best['predicted_class']}\n")
                f.write(f"Generations: {len(result.history)}\n")
                f.write(f"\nText:\n{best['text']}\n\n")

                # Victorian markers
                markers = count_victorian_markers(best['text'])
                f.write(f"Victorian Markers:\n")
                for k, v in markers.items():
                    f.write(f"  {k}: {v}\n")
                f.write("\n" + "-"*80 + "\n")

    print(f"✅ Saved: {ablation_dir}/best_individuals_all_conditions.txt")

    # 7. Create summary report
    print("\n7️⃣ Creating summary report...")

    with open(f"{ablation_dir}/ABLATION_STUDY_SUMMARY.txt", 'w') as f:
        f.write("="*80 + "\n")
        f.write("TASK 4: ABLATION STUDY SUMMARY\n")
        f.write("="*80 + "\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Conditions tested: BLIND, GUIDED, HYBRID\n")
        f.write(f"Seeds: [42, 123, 456]\n")
        f.write(f"Total experiments: {sum(len(v) for v in ablation_results.values())}\n\n")

        f.write("="*80 + "\n")
        f.write("RESULTS SUMMARY\n")
        f.write("="*80 + "\n\n")

        for condition, results in ablation_results.items():
            f.write(f"\n{condition.upper()} Condition:\n")
            f.write("-"*40 + "\n")

            final_probs = [r.best_individual['human_prob'] for r in results]
            f.write(f"Mean final fitness: {np.mean(final_probs)*100:.2f}% Human (± {np.std(final_probs)*100:.2f}%)\n")
            f.write(f"Best run: {np.max(final_probs)*100:.2f}% Human\n")
            f.write(f"Worst run: {np.min(final_probs)*100:.2f}% Human\n")

            # Check if any reached target
            reached_target = sum(1 for p in final_probs if p >= TARGET_FITNESS)
            f.write(f"Reached target ({TARGET_FITNESS*100:.0f}%): {reached_target}/3 runs\n")

        f.write("\n" + "="*80 + "\n")
        f.write("INTERPRETATION\n")
        f.write("="*80 + "\n\n")

        # Determine winner
        condition_means = {c: np.mean([r.best_individual['human_prob'] for r in results])
                          for c, results in ablation_results.items()}
        best_condition = max(condition_means, key=condition_means.get)

        f.write(f"Best performing condition: {best_condition.upper()}\n")
        f.write(f"Mean fitness: {condition_means[best_condition]*100:.2f}% Human\n\n")

        if best_condition == 'blind':
            f.write("INTERPRETATION: Blind mutations succeeded!\n")
            f.write("→ Detector has TRUE vulnerabilities unrelated to Victorian style\n")
            f.write("→ Random perturbations can fool it\n")
            f.write("→ Needs adversarial training\n")
        elif best_condition == 'guided':
            f.write("INTERPRETATION: Only guided mutations succeeded!\n")
            f.write("→ Detector is robust to blind search\n")
            f.write("→ Only vulnerable to domain experts with Victorian knowledge\n")
            f.write("→ This validates the original hypothesis\n")
        elif best_condition == 'hybrid':
            f.write("INTERPRETATION: Hybrid approach is best!\n")
            f.write("→ Combination exploits detector better than either alone\n")
            f.write("→ Detector has multiple weak points\n")
            f.write("→ Both random and guided perturbations help\n")

        f.write("\n" + "="*80 + "\n")
        f.write("FILES GENERATED\n")
        f.write("="*80 + "\n\n")
        f.write("  • ablation_fitness_comparison.png\n")
        f.write("  • ablation_victorian_markers_heatmap.png\n")
        f.write("  • convergence_analysis.csv\n")
        f.write("  • ablation_raw_data.csv\n")
        f.write("  • best_individuals_all_conditions.txt\n")
        f.write("  • ABLATION_STUDY_SUMMARY.txt (this file)\n")

        f.write("\n" + "="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")

    print(f"✅ Saved: {ablation_dir}/ABLATION_STUDY_SUMMARY.txt")

    print("\n" + "🎉"*40)
    print("🎉 ABLATION STUDY REPORT COMPLETE!")
    print("🎉"*40)
    print(f"\n📁 All results saved to: {ablation_dir}/")
    print("\n📊 Summary:")

    for condition, results in ablation_results.items():
        final_probs = [r.best_individual['human_prob'] for r in results]
        print(f"   {condition.upper():<8}: {np.mean(final_probs)*100:.2f}% ± {np.std(final_probs)*100:.2f}% Human")

    print("\n" + "🎉"*40)

print("✅ Report generation function defined!")
print("\n💡 Usage after ablation study:")
print("   generate_ablation_report(ablation_results, OUTPUT_DIR)")

---

# 🚀 QUICK START: Run Ablation Study

Copy-paste this code to run the complete ablation study after initial population is generated.

In [ ]:
# ============================================================================
# QUICK START: RUN COMPLETE ABLATION STUDY
# ============================================================================
#
# Prerequisites:
#   1. Run all setup cells (Gemini, model loading, dependencies)
#   2. Generate initial population (run "Phase 1: Naive Initial Prompts")
#
# This will run:
#   - Condition A (Blind): 3 seeds × ~5 min = ~15 min
#   - Condition B (Guided): 3 seeds × ~45 min = ~2.25 hours
#   - Condition C (Hybrid): 3 seeds × ~25 min = ~1.25 hours
#   TOTAL: ~3.5 hours
# ============================================================================

print("🚀 ABLATION STUDY QUICK START")
print("="*80)

# Check prerequisites
if 'initial_results' not in globals():
    print("❌ ERROR: Initial population not found!")
    print("   Please run the 'Generate Initial Population' cell first.")
elif blind_engine is None:
    print("❌ ERROR: Blind mutation engine not initialized!")
    print("   Please run the 'Install Additional Dependencies' cell.")
elif guided_engine is None:
    print("❌ ERROR: Guided mutation engine not initialized!")
    print("   Please check Gemini API configuration.")
else:
    print("✅ All prerequisites met!")
    print("\n📋 Configuration:")
    print(f"   Initial population: {len(initial_results)} individuals")
    print(f"   Conditions: BLIND, GUIDED, HYBRID")
    print(f"   Seeds: [42, 123, 456]")
    print(f"   Generations: {NUM_GENERATIONS}")
    print(f"   Population size: {POPULATION_SIZE}")
    print(f"   Target: {TARGET_FITNESS*100:.0f}% Human")

    print("\n⏱️ Estimated runtime: ~3.5 hours")
    print("\n" + "="*80)
    print("Ready to run! Execute the following commands:")
    print("="*80)
    print("""
# Run full ablation study
ablation_results = run_full_ablation_study(
    initial_population=[r['text'] for r in initial_results],
    seeds=[42, 123, 456],
    conditions=['blind', 'guided', 'hybrid']
)

# Generate complete analysis report
generate_ablation_report(ablation_results, OUTPUT_DIR)
    """)
    print("="*80)
    print("\n💡 TIP: You can run individual conditions by modifying conditions list")
    print("   Example: conditions=['blind'] for quick test (~15 min)")
    print("\n💡 TIP: Reduce seeds for faster testing")
    print("   Example: seeds=[42] runs 1 experiment per condition (~1.5 hours)")

print("\n✅ Quick start guide ready!")

---

# 🧪 TEST: Verify Mutation Engines

Run this cell to test all three mutation engines on sample text.

In [ ]:
# Test all mutation engines
print("🧪 TESTING MUTATION ENGINES")
print("="*80)

test_text = """The detective investigated the mysterious crime scene in London.
Evidence was carefully examined. The case presented several intriguing clues
that would eventually lead to the solution."""

print(f"\n📝 Original text:\n{test_text}\n")

# Test 1: Blind mutations
if blind_engine is not None:
    print("\n" + "─"*80)
    print("🔬 CONDITION A: BLIND MUTATIONS")
    print("─"*80)

    for mutation_type in ['synonym_replacement', 'sentence_shuffle', 'word_deletion']:
        print(f"\n{mutation_type}:")
        try:
            mutated = blind_engine._apply_mutation(test_text, mutation_type)
            print(f"  → {mutated[:150]}...")
        except Exception as e:
            print(f"  ⚠️ Error: {e}")
else:
    print("\n⚠️ Blind engine not available")

# Test 2: Guided mutations
if guided_engine is not None:
    print("\n" + "─"*80)
    print("🎯 CONDITION B: GUIDED MUTATIONS")
    print("─"*80)

    # Test one strategy
    print("\nTesting 'temporal_shift' strategy...")
    try:
        mutated, strategy = guided_engine.mutate(test_text, generation=1)
        print(f"  Strategy used: {strategy}")
        print(f"  → {mutated[:150]}...")
    except Exception as e:
        print(f"  ⚠️ Error: {e}")
else:
    print("\n⚠️ Guided engine not available")

# Test 3: Hybrid
if hybrid_engine is not None:
    print("\n" + "─"*80)
    print("⚡ CONDITION C: HYBRID MUTATIONS")
    print("─"*80)

    print("\nTesting hybrid (will randomly choose blind or guided)...")
    try:
        mutated, mutation_type = hybrid_engine.mutate(test_text, generation=1)
        print(f"  Mutation type: {mutation_type}")
        print(f"  → {mutated[:150]}...")
    except Exception as e:
        print(f"  ⚠️ Error: {e}")
else:
    print("\n⚠️ Hybrid engine not available")

print("\n" + "="*80)
print("✅ Mutation engine tests complete!")
print("="*80)

In [27]:
kablation_results = run_full_ablation_study(
    initial_population=[r['text'] for r in initial_results],
    seeds=[42, 123, 456],
    conditions=['blind', 'guided', 'hybrid']
)

generate_ablation_report(ablation_results, OUTPUT_DIR)


🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
🔬 STARTING FULL ABLATION STUDY
🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬

Conditions: BLIND, GUIDED, HYBRID
Seeds: [42, 123, 456]
Total experiments: 9
Initial population: 10 individuals

🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬


🧪 CONDITION: BLIND

[Experiment 1/9] BLIND (seed=42)

🔬 RUNNING ABLATION EXPERIMENT
Condition: BLIND
Seed: 42
Initial population: 10 individuals
Generations: 10

────────────────────────────────────────────────────────────────────────────────
Generation 1/10 (BLIND)
────────────────────────────────────────────────────────────────────────────────
   Best: 0.0506 (0.06% Human, AI)
   Avg:  0.0502
   🧬 Mutating...
   ✓ Generation 2 ready

────────────────────────────────────────────────────────────────────────────────
Generation 2/10 (BLIND)
────────────────────────────────────────────────────────────────────────────────
   Best: 0.0509 (0.09% Human, AI)
   Avg:  0.0506
   🧬 Mutating...
   ✓ Generation 3 ready

───────────

NameError: name 'generate_ablation_report' is not defined

---

# 📖 COMPLETE USAGE GUIDE

## Option 1: Run Original GA
```python
# Just run cells sequentially - no changes needed!
# Your existing GA loop uses guided mutations by default
```

## Option 2: Run Single Ablation Condition (Quick Test)
```python
# After generating initial population, run ONE condition:

# Test blind mutations (~5 min)
blind_result = run_ablation_experiment(
    condition='blind',
    seed=42,
    initial_population=[r['text'] for r in initial_results]
)

# Test guided mutations (~45 min)
guided_result = run_ablation_experiment(
    condition='guided',
    seed=42,
    initial_population=[r['text'] for r in initial_results]
)

# Test hybrid (~25 min)
hybrid_result = run_ablation_experiment(
    condition='hybrid',
    seed=42,
    initial_population=[r['text'] for r in initial_results]
)
```

## Option 3: Run Full Ablation Study (3 conditions × 3 seeds)
```python
# Run complete ablation study
ablation_results = run_full_ablation_study(
    initial_population=[r['text'] for r in initial_results],
    seeds=[42, 123, 456],
    conditions=['blind', 'guided', 'hybrid']
)

# Generate comprehensive analysis
generate_ablation_report(ablation_results, OUTPUT_DIR)
```

## Option 4: Custom Configuration
```python
# Run specific conditions with custom seeds
ablation_results = run_full_ablation_study(
    initial_population=[r['text'] for r in initial_results],
    seeds=[42],  # Single seed for faster testing
    conditions=['blind', 'guided']  # Only 2 conditions
)

# Or run experiments manually
results_blind = []
for seed in [42, 123]:
    result = run_ablation_experiment('blind', seed, initial_pop)
    results_blind.append(result)

results_guided = []
for seed in [42, 123]:
    result = run_ablation_experiment('guided', seed, initial_pop)
    results_guided.append(result)

ablation_results = {'blind': results_blind, 'guided': results_guided}
generate_ablation_report(ablation_results, OUTPUT_DIR)
```

## 📊 Expected Outputs

All results saved to `{OUTPUT_DIR}/ablation_study/`:

1. **ablation_fitness_comparison.png** - Line plot + boxplot
2. **ablation_victorian_markers_heatmap.png** - 3 heatmaps (one per condition)
3. **convergence_analysis.csv** - Generations to reach 70% Human
4. **ablation_raw_data.csv** - All generation data (condition, seed, gen, fitness)
5. **best_individuals_all_conditions.txt** - Final evolved texts with analysis
6. **ABLATION_STUDY_SUMMARY.txt** - Complete interpretation report
---